---
### Follow the Workers: Study 1 diagnostics, Study 2 (post-hoc iteration) and the forward test
---

---
##### Imports
Project modules; Study 1 is only ever imported read-only inside data.py.

In [1]:
import hashlib
import json
import re
import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import params as P
import data as D
import stats as S
import plots as G

# numpy 2.0 with Apple Accelerate raises spurious "encountered in matmul" warnings; Gate 1 below
# shows results match the frozen study to ~1e-16, so they are silenced (only that message).
warnings.filterwarnings('ignore', message='.*encountered in matmul', category=RuntimeWarning)
pd.set_option('display.width', 160)
for folder in [P.OUT, P.OUT1, P.OUT2, P.TABLES1, P.TABLES2, P.FIGS, P.TEX, P.FORWARD, P.SPECS]:
    folder.mkdir(parents=True, exist_ok=True)


def utc_now():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')


def sha256(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()


def stop(reason):
    """A failed gate: record it and halt the notebook (project rule 6: a failed gate halts the run)."""
    (P.OUT / 'STOP.md').write_text(f'# STOP\n\n{utc_now()}\n\n{reason}\n')
    raise RuntimeError(reason)


def frozen_fingerprint():
    files = sorted(p for p in P.FROZEN_REPO.rglob('*') if p.is_file() and '.git' not in p.relative_to(P.FROZEN_REPO).parts)
    h = hashlib.sha256()
    for p in files:
        h.update(str(p.relative_to(P.FROZEN_REPO)).encode())
        h.update(hashlib.sha256(p.read_bytes()).digest())
    return h.hexdigest(), len(files)

---
##### Data
Public French returns, current FRED labor data with fixed publication lags, and Study 1's saved outputs.

In [2]:
R = D.load_returns()                 # 13 group returns, excess, relative, factors
F, T = D.signals()                   # standardised F1..F5, W and tightness T, by earning month
F['A0'] = (F['F1'] + F['F2'] - F['F4'] + F['F5']) / 4      # preregistered fixed rule
F['IndMom'] = D.industry_momentum(R['total'])             # industry momentum 12-1 (months m-12..m-2)
FP, FD = D.frozen()                  # frozen params.py and data.py (read-only)
print('returns', R['total'].index.min(), '->', R['total'].index.max(),
      '| signals', F['W'].dropna(how='all').index.min(), '->', F['W'].dropna(how='all').index.max())

returns 1963-07 -> 2026-08 | signals 2004-04 -> 2026-11


In [3]:
# Which data vintage is used (iteration task, Part 1 step 2), with the evidence for the choice.
panels_in_repo = sorted(str(p.relative_to(P.FROZEN_REPO)) for p in P.FROZEN_REPO.rglob('*.parquet'))
env_key = (P.ROOT / '.env').exists() and 'FRED_API_KEY' in (P.ROOT / '.env').read_text()
vintage_check = {}
for variant in ['FIXEDLAG', 'ASOF', 'REVISED', 'FIRST']:
    saved = D.saved_csv(f'{variant}_A0_scores.csv')
    ours = F['A0'].reindex(saved.index)
    vintage_check[variant] = {
        'pooled_corr': float(np.corrcoef(ours.to_numpy().ravel(), saved.to_numpy().ravel())[0, 1]),
        'mean_monthly_rank_corr': float(S.rank_ic(ours, saved).mean()),
        'same_top3_share': float(np.mean([set(ours.loc[m].nlargest(3).index) == set(saved.loc[m].nlargest(3).index) for m in saved.index]))}
vintage = pd.DataFrame(vintage_check).T
vintage.index.name = 'saved A0 variant'
res = json.loads((P.SEALED / 'results.json').read_text())
ic_a3 = {v: res['variants'][v]['A3']['IC']['1']['mean'] for v in ['ASOF', 'FIXEDLAG']}
sr_a3 = {v: res['variants'][v]['A3']['sharpe'] for v in ['ASOF', 'FIXEDLAG']}
choice = 'a' if panels_in_repo else 'b' if env_key else 'c'
text = f"""# Data vintage used in this iteration

Written by analysis/run.ipynb on {utc_now()}.

**Choice: option ({choice})** of the iteration task, Part 1 step 2.

- (a) As-known (ASOF) panels saved in the frozen repo: **not available**. The repo contains {len(panels_in_repo)} parquet files; its package_manifest.json lists the panels as omitted.
- (b) ALFRED vintages with `analysis/.env` FRED_API_KEY: **not available** (no key file).
- (c) Revised FRED data downloaded 3 Oct 2026 from fredgraph.csv, with fixed publication lags (JOLTS month d-2, CES month d-1, tightness month d-2): **used** for every historical result in this iteration.

**Known exception, found in the 3 Oct 2026 code review and left uncorrected.** The fixed lags assume the usual BLS release calendar, which the federal shutdown of 1 Oct to 12 Nov 2025 broke. Checked against ALFRED vintages by the reviewer: at the 31 Oct 2025 decision the latest published CES month was Aug 2025 (the rule assumes Sep); at the 28 Nov 2025 decision the latest CES month was Sep 2025 and the latest JOLTS month Aug 2025 (the rule assumes Oct and Sep). Earning months Nov 2025 (W, F5) and Dec 2025 (F1-F5, W, T) therefore use values released after the decision. The frozen study's as-known (ASOF) variant handled this correctly; its FIXEDLAG diagnostic shares the issue. Correcting it would change v2/v3 numbers after their run logs were frozen, so it is logged in docs/OPEN_ISSUES.md instead; any future run should apply the ALFRED availability rule.

The forward-test book (Part 3) does not use (c). It uses ALFRED values *as published on {P.VINTAGE_DATE}*, downloaded without a key from alfredgraph.csv, so it only contains data known before the 30 Sep 2026 decision.

## How close is (c) to the frozen study's own signals?

Our rebuilt A0 composite (same transforms) against the frozen study's saved sealed A0 scores, Nov 2014 to Aug 2026:

{G.markdown(vintage.round(3))}

Our data is the FIXEDLAG construction (revised values, fixed lags), and it reproduces the frozen FIXEDLAG scores almost exactly. As-known (ASOF) data picks the same top three industries in only {vintage_check['ASOF']['same_top3_share']:.0%} of months, so revisions do change portfolio membership. The frozen study's primary A3 had the same test IC under ASOF ({ic_a3['ASOF']:+.4f}) and FIXEDLAG ({ic_a3['FIXEDLAG']:+.4f}) and a test Sharpe of {sr_a3['ASOF']:+.2f} against {sr_a3['FIXEDLAG']:+.2f} (results.json), so the verdict is not sensitive to this choice, but monthly holdings and the size of the loss are. Every historical number in this iteration is therefore labelled as computed on revised data.
"""
(P.OUT1 / 'data_vintage.md').write_text(text)
vintage.round(4)

,pooled_corr,mean_monthly_rank_corr,same_top3_share
saved A0 variant,,,
FIXEDLAG,0.9983,0.9922,0.9085
ASOF,0.9181,0.8635,0.4155
REVISED,0.9912,0.9793,0.8592
FIRST,0.8506,0.7796,0.2394


---
##### Gate 1
Reproduce the sealed A0 net returns from its saved scores with the frozen backtest, and the 13 group returns against every saved ledger.

In [4]:
checks = []

# 1. The frozen repo is unchanged since the baseline fingerprint.
fingerprint, n_files = frozen_fingerprint()
checks.append(('Frozen repo unchanged (SHA-256 over all files)', f'{n_files} files, {fingerprint[:12]}...',
               f'{P.FROZEN_FINGERPRINT[1]} files, {P.FROZEN_FINGERPRINT[0][:12]}...', (fingerprint, n_files) == P.FROZEN_FINGERPRINT))

# 2. Portfolio parameters equal the frozen study's.
same = all(FP.PORTFOLIO[k] == v for k, v in P.PORTFOLIO.items())
checks.append(('Portfolio parameters equal frozen params.PORTFOLIO', str(P.PORTFOLIO), 'identical', same))

# 3. Sealed A0 net monthly returns from the saved A0 scores, through the frozen backtest.
saved_ledger = D.saved_csv('ASOF_A0_ledger.csv')
ledger, _ = D.backtest(D.saved_csv('ASOF_A0_scores.csv'), R)
for col in ['net', 'gross', 'cost', 'turnover']:
    diff = float((ledger[col] - saved_ledger[col]).abs().max())
    checks.append((f'Sealed A0 {col}: max |ours - saved| over {len(ledger)} months', f'{diff:.2e}', '< 1e-6', diff < 1e-6))

# 4. Group returns: saved weights x our group excess returns + hedge = saved gross, every saved ledger.
worst, count = 0.0, 0
for study in [P.SONNET, P.OPUS]:
    sealed = sorted((study / 'outputs/sealed/initial').glob('*_weights.csv'))
    research = sorted((study / 'outputs').glob('research_A*_weights.csv'))
    for path in sealed + research:
        folder = 'outputs/sealed/initial' if path in sealed else 'outputs'
        w = D.saved_csv(path.name, study, folder)
        g = D.saved_csv(path.name.replace('_weights', '_ledger'), study, folder)
        groups = [c for c in w.columns if c != 'market']
        recon = (w[groups] * R['excess'].loc[w.index, groups]).sum(axis=1) + w['market'] * R['factors'].loc[w.index, 'Mkt-RF']
        worst = max(worst, float((recon - g.gross).abs().max()))
        count += 1
checks.append((f'13 group returns vs {count} saved ledgers (both studies, sealed + research)', f'{worst:.2e}', '< 1e-6', worst < 1e-6))

# 5. Our estimators reproduce the frozen results.json (IC, HAC t, Sharpe, factor regression).
res = json.loads((P.SEALED / 'results.json').read_text())
worst_est = 0.0
for s in ['A0', 'A1', 'A1-T', 'A3', 'A4']:
    l, sc = D.saved_csv(f'ASOF_{s}_ledger.csv'), D.saved_csv(f'ASOF_{s}_scores.csv')
    ic = S.rank_ic(sc, R['relative'])
    fa = S.factor_attribution(l.net, R)
    ref, ref_a = res['variants']['ASOF'][s], res['attribution'][s]
    diffs = [float((ic - D.saved_csv(f'ASOF_{s}_ic.csv')['ic']).abs().max()),
             abs(S.hac(ic, lags=6)['t'] - ref['IC']['1']['t']), abs(S.perf(l)['sharpe'] - ref['sharpe']),
             max(abs(fa['coef'][k] - ref_a['coefficients'][k]) for k in ref_a['coefficients']),
             max(abs(fa['tstat'][k] - ref_a['t_statistics'][k]) for k in ref_a['t_statistics'])]
    worst_est = max(worst_est, max(diffs))
checks.append(('Estimators vs results.json (IC, IC t, Sharpe, factor coefs and t; A0-A4)', f'{worst_est:.2e}', '< 1e-8', worst_est < 1e-8))

gate1 = pd.DataFrame(checks, columns=['check', 'value', 'threshold', 'pass'])
gate1

,check,value,threshold,pass
0,Frozen repo unchanged (SHA-256 over all files),"3067 files, eb109be96917...","3067 files, eb109be96917...",True
1,Portfolio parameters equal frozen params.PORTF...,"{'longs': 3, 'shorts': 3, 'holding': 3, 'risk_...",identical,True
2,Sealed A0 net: max |ours - saved| over 142 months,9.97e-17,< 1e-6,True
3,Sealed A0 gross: max |ours - saved| over 142 m...,9.71e-17,< 1e-6,True
4,Sealed A0 cost: max |ours - saved| over 142 mo...,9.94e-17,< 1e-6,True
5,Sealed A0 turnover: max |ours - saved| over 14...,2.22e-16,< 1e-6,True
6,13 group returns vs 50 saved ledgers (both stu...,1.51e-16,< 1e-6,True
7,"Estimators vs results.json (IC, IC t, Sharpe, ...",3.55e-15,< 1e-8,True


In [5]:
passed = bool(gate1['pass'].all())
lines = [f'# Gate 1 validation\n\nRun {utc_now()} by analysis/run.ipynb. **Result: {"PASS" if passed else "FAIL"}**\n',
         'Every check uses the frozen study\'s saved files in `study1_preregistered/` (read-only) and public Ken French data '
         '(CRSP 202608 build) in `analysis/data/french/`. The backtest is the frozen `data.backtest`, imported read-only.\n',
         G.markdown(gate1, index=False), '',
         '## Informational: rebuilt labor signals vs the frozen study\'s saved A0 scores', '',
         'Not part of the gate (our labor data is revised FRED, not the frozen as-known panels). See `data_vintage.md`.', '',
         G.markdown(vintage.round(4)), '']
(P.OUT1 / 'validation.md').write_text('\n'.join(lines))
if not passed:
    stop('Gate 1 failed. See analysis/output/study1/validation.md:\n\n' + G.markdown(gate1, index=False))
print('Gate 1:', 'PASS' if passed else 'FAIL')

Gate 1: PASS


---
##### Portfolio equation
The frozen score-to-weight mapping, written as LaTeX for the report.

In [6]:
equation = r"""% Score-to-weight mapping of the frozen study's backtest (study1_preregistered/outputs/follow_the_workers/data.py::backtest).
% Written by analysis/run.ipynb. Parameters: analysis/params.py PORTFOLIO (checked equal to the frozen params in Gate 1).
\begin{align}
\tau_{m,g} &= \begin{cases} +\tfrac13 & g \in \text{top 3 of } s_{m,\cdot}\\ -\tfrac13 & g \in \text{bottom 3 of } s_{m,\cdot}\\ 0 & \text{otherwise}\end{cases}
  \qquad b_{m} = \frac{1}{H}\sum_{j=0}^{H-1} \tau_{m-j}, \quad H=3, \label{eq:tranche}\\
h_{m} &= -\sum_{g} b_{m,g}\,\hat\beta_{m,g}, \qquad x_m = (b_{m,1},\dots,b_{m,13},\,h_m), \label{eq:hedge}\\
k_m &= \min\!\left(\frac{0.10}{\sqrt{12\,x_m^{\top}\hat\Sigma_m x_m}},\ \frac{2}{\lVert x_m\rVert_1}\right), \qquad w_m = k_m\,x_m, \label{eq:scale}\\
r^{\mathrm{net}}_{m} &= \sum_{g} w_{m,g}\, r^{e}_{m,g} + w_{m,M}\,\mathrm{MKT}_m
  - c\sum_{g}\lvert w_{m,g}-\tilde w_{m,g}\rvert - c_h\,\lvert w_{m,M}-\tilde w_{m,M}\rvert. \label{eq:net}
\end{align}
where $s_{m,g}$ is the score of group $g$ known at the end of month $m-1$, $\hat\beta_{m,g}$ and the Ledoit--Wolf covariance
$\hat\Sigma_m$ (13 groups plus the market) use the 60 months ending in $m-2$ (at least 36), $r^{e}_{m,g}$ is the group's excess
return, $\tilde w$ are the previous weights after drift, $c=10$\,bp and $c_h=2$\,bp."""
(P.TEX / 'portfolio_equation.tex').write_text(equation + '\n')
print((P.TEX / 'portfolio_equation.tex').read_text()[:200])

% Score-to-weight mapping of the frozen study's backtest (study1_preregistered/outputs/follow_the_workers/data.py::backtest).
% Written by analysis/run.ipynb. Parameters: analysis/params.py PORTFOLIO 


---
##### Diagnostic helpers
Windows, IC statistics and a registry that feeds diagnostics_v2.md; everything below is POST-HOC unless labelled CONFIRMATORY.

In [7]:
SIGNALS = ['F1', 'F2', 'F3', 'F4', 'F5', 'W', 'A0', 'IndMom']
months = lambda w: pd.period_range(*P.WINDOWS[w], freq='M')
REGISTRY = []


def ic_stats(score, target, window_months, lags=P.HAC_LAGS):
    """Mean monthly rank IC in a window with its Newey-West s.e., t and number of months."""
    ic = S.rank_ic(score.reindex(window_months), target.reindex(window_months))
    h = S.hac(ic, lags=lags)
    return h['mean'], h['se'], h['t'], h['n']


def register(item, title, label, name, sources, df, note=''):
    REGISTRY.append(dict(item=item, title=title, label=label, name=name, sources=sources, df=df, note=note))


def ic_text(ic, t, digits=3):
    return [f'{a:+.{digits}f} ({b:+.2f})' if np.isfinite(a) else 'n/a' for a, b in zip(ic, t)]

---
##### D1 Single-signal IC
Rank IC of each labor signal, the preregistered composite and industry momentum against next-month relative returns.

In [8]:
rows = {}
for s in SIGNALS:
    row = {}
    for w in ['research', 'test', 'full']:
        m, se, t, n = ic_stats(F[s], R['relative'], months(w))
        row.update({f'{w}_ic': m, f'{w}_se': se, f'{w}_t': t, f'{w}_n': n})
    rows[s] = row
d1 = pd.DataFrame(rows).T
d1.index.name = 'signal'
d1_show = pd.DataFrame({'Research 2004--14': ic_text(d1.research_ic, d1.research_t),
                        'Test 2014--26': ic_text(d1.test_ic, d1.test_t),
                        'Full 2004--26': ic_text(d1.full_ic, d1.full_t)},
                       index=pd.Index([G.SIGNAL_LABELS[s] for s in d1.index], name='Signal'))
G.write_table(d1, 'd1_signal_ic', label=P.POSTHOC, display=d1_show, align='lrrr')
G.ic_signals(d1)
register('D1', 'Single-signal rank IC (t in brackets), next-month relative return', P.POSTHOC, 'd1_signal_ic',
         'analysis/data/fred (revised, fixed lags), analysis/data/french', d1_show,
         'IC = Spearman correlation across 13 groups each month; t from Newey-West with 6 lags. '
         'Industry momentum uses months m-12..m-2 (skips m-1), matching the frozen information rule. '
         'The 3 Oct baseline row "IndMom 12-1" used m-12..m-1 and is superseded; its "IndMom 12-2" row equals this definition. '
         f'Sensitivity: W standardisation history now starts at the frozen FEATURE_START (decision Apr 2002). The 3 Oct baseline '
         f'started it in Jan 2001 and found research/test IC +0.039/+0.041; here {d1.loc["W", "research_ic"]:+.3f}/{d1.loc["W", "test_ic"]:+.3f}. '
         'The full-sample IC is unchanged, but the research/test split is not stable to this choice.')
d1.round(3)

,research_ic,research_se,research_t,research_n,test_ic,test_se,test_t,test_n,full_ic,full_se,full_t,full_n
signal,,,,,,,,,,,,
F1,0.001,0.024,0.024,125.0,-0.011,0.026,-0.436,142.0,-0.005,0.018,-0.298,268.0
F2,0.012,0.025,0.481,125.0,0.013,0.028,0.478,142.0,0.013,0.019,0.710,268.0
F3,0.014,0.028,0.495,125.0,0.002,0.021,0.115,142.0,0.009,0.017,0.502,268.0
F4,0.033,0.030,1.070,125.0,0.026,0.028,0.932,142.0,0.028,0.020,1.376,268.0
F5,-0.012,0.021,-0.555,125.0,-0.007,0.020,-0.340,142.0,-0.010,0.015,-0.674,268.0
W,0.029,0.019,1.537,125.0,0.049,0.024,2.036,142.0,0.039,0.016,2.502,268.0
A0,-0.023,0.026,-0.879,125.0,-0.006,0.026,-0.212,142.0,-0.013,0.018,-0.689,268.0
IndMom,0.036,0.029,1.254,125.0,0.003,0.028,0.108,142.0,0.020,0.020,0.990,268.0


---
##### D2 IC by horizon
Rank IC of each signal against 1-, 3-, 6- and 12-month compounded relative returns, full sample.

In [9]:
d2_ic, d2_t = {}, {}
for h in P.HORIZONS:
    target = D.horizon_target(R['total'], h)
    for s in SIGNALS:
        m, se, t, n = ic_stats(F[s], target, months('full'), lags=h + 5)
        d2_ic.setdefault(s, {})[h] = m
        d2_t.setdefault(s, {})[h] = t
d2_ic, d2_t = pd.DataFrame(d2_ic).T, pd.DataFrame(d2_t).T
d2 = pd.concat([d2_ic.add_prefix('ic_h'), d2_t.add_prefix('t_h')], axis=1)
d2.index.name = 'signal'
d2_show = pd.DataFrame({f'{h}m': ic_text(d2_ic[h], d2_t[h]) for h in P.HORIZONS},
                       index=pd.Index([G.SIGNAL_LABELS[s] for s in d2_ic.index], name='Signal'))
G.write_table(d2, 'd2_horizon_ic', label=P.POSTHOC, display=d2_show, align='lrrrr')
G.horizon(d2_ic, d2_t)
register('D2', 'Rank IC by horizon h (t in brackets), May 2004-Aug 2026', P.POSTHOC, 'd2_horizon_ic',
         'analysis/data/fred, analysis/data/french', d2_show,
         'Target = h-month compounded total return demeaned across groups; Newey-West lags h+5 (frozen convention).')
d2.round(3)

,ic_h1,ic_h3,ic_h6,ic_h12,t_h1,t_h3,t_h6,t_h12
signal,,,,,,,,
F1,-0.005,0.007,-0.011,-0.039,-0.307,0.226,-0.284,-0.918
F2,0.013,0.002,0.002,-0.062,0.703,0.069,0.084,-2.081
F3,0.009,-0.013,-0.022,-0.065,0.500,-0.553,-0.775,-1.727
F4,0.028,0.005,0.000,-0.036,1.370,0.169,0.009,-1.277
F5,-0.010,-0.027,-0.049,-0.057,-0.672,-1.244,-1.689,-1.861
W,0.039,0.037,0.037,0.026,2.506,1.769,1.277,0.633
A0,-0.013,-0.002,-0.021,-0.053,-0.689,-0.085,-0.609,-1.580
IndMom,0.020,0.025,0.048,0.029,0.990,0.801,1.237,0.635


---
##### D3 Tightness
The tightness path, why expanding terciles fail, and signal IC in tightening vs loosening months.

In [10]:
G.timeline(T)
# Regime balance: the frozen expanding-tercile rule vs a 12-month change rule.
low, high = T.expanding(min_periods=P.STD_MIN_MONTHS).quantile(1 / 3), T.expanding(min_periods=P.STD_MIN_MONTHS).quantile(2 / 3)
tercile = pd.Series('middle', index=T.index).where(T.notna())
tercile[T <= low] = 'low'
tercile[T >= high] = 'high'
tercile[low.isna()] = np.nan
dT = T - T.shift(12)
balance = {}
for w in ['research', 'test']:
    idx = months(w)
    counts = tercile.reindex(idx).value_counts()
    balance[w] = {'months': len(idx), 'expanding low': int(counts.get('low', 0)), 'expanding middle': int(counts.get('middle', 0)),
                  'expanding high': int(counts.get('high', 0)), 'T rising (12m)': int((dT.reindex(idx) > 0).sum()),
                  'T falling (12m)': int((dT.reindex(idx) <= 0).sum()), 'V/U > 1': int((T.reindex(idx) > 0).sum()),
                  'T min': float(T.reindex(idx).min()), 'T max': float(T.reindex(idx).max())}
d3_balance = pd.DataFrame(balance).T
d3_balance.index.name = 'period'
d3_balance_show = pd.DataFrame({w: [f'{int(v)}' if k not in ('T min', 'T max') else f'{v:+.2f}' for k, v in d3_balance.loc[w].items()]
                                for w in d3_balance.index}, index=pd.Index(d3_balance.columns, name='Months in regime'))
d3_balance_show.columns = ['Research 2004--14', 'Test 2014--26']
G.write_table(d3_balance, 'd3_regime_balance', label=P.POSTHOC, display=d3_balance_show, align='lrr')
# IC by tightening / loosening, full sample (and by period in the CSV).
rows, by_period = {}, []
for s in SIGNALS:
    ic = S.rank_ic(F[s].reindex(months('full')), R['relative'].reindex(months('full')))
    up, down = dT.reindex(ic.index) > 0, dT.reindex(ic.index) <= 0
    r, f = S.hac(ic.where(up)), S.hac(ic.where(down))
    rows[s] = {'rising_ic': r['mean'], 'rising_se': r['se'], 'rising_t': r['t'], 'n_rising': r['n'],
               'falling_ic': f['mean'], 'falling_se': f['se'], 'falling_t': f['t'], 'n_falling': f['n'],
               'difference': r['mean'] - f['mean']}
    for w in ['research', 'test']:
        sub = ic.reindex(months(w))
        for state, mask in [('rising', up), ('falling', down)]:
            hh = S.hac(sub.where(mask.reindex(sub.index, fill_value=False)))
            by_period.append({'signal': s, 'period': w, 'state': state, 'ic': hh['mean'], 't': hh['t'], 'n': hh['n']})
d3 = pd.DataFrame(rows).T
d3.index.name = 'signal'
pd.DataFrame(by_period).to_csv(P.table_path('d3_tightness_ic_by_period'), index=False)
d3_show = pd.DataFrame({f'T rising ({int(d3.n_rising.iloc[0])} m)': ic_text(d3.rising_ic, d3.rising_t),
                        f'T falling ({int(d3.n_falling.iloc[0])} m)': ic_text(d3.falling_ic, d3.falling_t),
                        'Difference': [f'{v:+.3f}' for v in d3.difference]},
                       index=pd.Index([G.SIGNAL_LABELS[s] for s in d3.index], name='Signal'))
G.write_table(d3, 'd3_tightness_ic', label=P.POSTHOC, display=d3_show, align='lrrr')
G.tightness(d3)
register('D3a', 'Tightness regime balance (months per regime)', P.POSTHOC, 'd3_regime_balance',
         'analysis/data/fred (JTSJOL, UNEMPLOY)', d3_balance_show,
         'Expanding terciles replicate the frozen tightness_regimes() rule (expanding quantiles, 24-month minimum).')
register('D3b', 'Rank IC in months when T rose vs fell over the previous 12 months, May 2004-Aug 2026', P.POSTHOC,
         'd3_tightness_ic', 'analysis/data/fred, analysis/data/french; per-period split in d3_tightness_ic_by_period.csv', d3_show)
d3_balance

,months,expanding low,expanding middle,expanding high,T rising (12m),T falling (12m),V/U > 1,T min,T max
period,,,,,,,,,
research,125.0,49.0,26.0,50.0,93.0,32.0,0.0,-1.878192,-0.304915
test,142.0,3.0,6.0,133.0,88.0,54.0,75.0,-1.611780,0.712436


---
##### D4 Cumulative test returns
CONFIRMATORY ledgers of the preregistered strategies, plus industry momentum through the same backtest (post-hoc).

In [11]:
conf_ledgers = {s: D.saved_csv(f'ASOF_{s}_ledger.csv') for s in ['A0', 'A1', 'A3', 'A4']}
mom_ledger, mom_weights = D.backtest(F['IndMom'].reindex(months('full')), R)   # continuous 2004-05..2026-08
test = months('test')
assert all(l.index.equals(test) for l in conf_ledgers.values())
names = {'A0': 'A0 fixed rule', 'A1': 'A1 ridge', 'A3': 'A3 primary', 'A4': 'A4 island agents'}
rows = {}
for s, l in conf_ledgers.items():
    rows[names[s]] = {'label': P.CONFIRMATORY, **S.perf(l)}
rows['Industry momentum 12-1'] = {'label': P.POSTHOC, **S.perf(mom_ledger.loc[test])}
d4 = pd.DataFrame(rows).T[['label', 'n', 'sharpe', 'net_ann', 'vol', 'max_dd', 'hit', 'turnover', 'cost_ann']]
d4.index.name = 'strategy'
series = pd.DataFrame({names[s]: l.net for s, l in conf_ledgers.items()} | {'Industry momentum 12-1': mom_ledger.net.loc[test]})
series.to_csv(P.table_path('d4_test_net_returns'))
d4_show = d4.copy()
for c, f in {'sharpe': '{:+.2f}', 'net_ann': '{:+.1%}', 'vol': '{:.1%}', 'max_dd': '{:.1%}', 'hit': '{:.0%}', 'turnover': '{:.2f}', 'cost_ann': '{:.2%}'}.items():
    d4_show[c] = [f.format(v) for v in d4[c]]
d4_show.columns = ['Label', 'Months', 'Sharpe', 'Net p.a.', 'Vol.', 'Max DD', 'Hit rate', 'Turnover', 'Costs p.a.']
G.write_table(d4, 'd4_test_performance', label=f'{P.CONFIRMATORY} (A0-A4) / {P.POSTHOC} (momentum)',
              display=d4_show.drop(columns=['Label', 'Months', 'Hit rate']), align='lrrrrrr')
G.cumulative({names[s]: l.net for s, l in conf_ledgers.items()},
             {'Industry momentum 12-1 (post-hoc)': (mom_ledger.net.loc[test], ':')})
register('D4', 'Test-period performance, Nov 2014-Aug 2026', 'CONFIRMATORY (A0-A4) / POST-HOC (momentum)', 'd4_test_performance',
         'study1_preregistered/.../sealed/initial/ASOF_*_ledger.csv; momentum: frozen backtest on analysis/data/french', d4_show,
         'Annualised mean = 12 x monthly mean net excess return; Sharpe on net excess returns; drawdown of funded wealth (RF + net). '
         'Momentum backtest runs continuously from May 2004; the test window is a slice.')
d4

,label,n,sharpe,net_ann,vol,max_dd,hit,turnover,cost_ann
strategy,,,,,,,,,
A0 fixed rule,CONFIRMATORY,142,-0.359456,-0.034706,0.096553,-0.402753,0.464789,0.831768,0.01012
A1 ridge,CONFIRMATORY,142,-0.507481,-0.03988,0.078585,-0.377362,0.450704,0.910112,0.011066
A3 primary,CONFIRMATORY,142,-0.584495,-0.045877,0.078489,-0.410629,0.443662,0.913789,0.01111
A4 island agents,CONFIRMATORY,142,-0.786325,-0.060973,0.077542,-0.532903,0.464789,1.105963,0.013452
Industry momentum 12-1,POST-HOC,142,0.03484,0.003287,0.094335,-0.208265,0.471831,0.531534,0.00651


---
##### D5 Industry P&L
Where the primary A3 made and lost money in the test period.

In [12]:
w3, l3 = D.saved_csv('ASOF_A3_weights.csv'), D.saved_csv('ASOF_A3_ledger.csv')
groups = list(P.GROUPS)
contrib = (w3[groups] * R['excess'].loc[w3.index, groups]).sum()
pnl = pd.concat([contrib.rename(index=P.NAMES),
                 pd.Series({'Market hedge': float((w3['market'] * R['factors'].loc[w3.index, 'Mkt-RF']).sum()),
                            'Trading costs': -float(l3.cost.sum() + l3.borrow_financing.sum())})])
assert abs(pnl.sum() - l3.net.sum()) < 1e-10, 'P&L decomposition does not add up to net'
d5 = pd.DataFrame({'contribution': pnl,
                   'months_long_share': pd.Series({P.NAMES[g]: (w3[g] > 1e-9).mean() for g in groups}),
                   'months_short_share': pd.Series({P.NAMES[g]: (w3[g] < -1e-9).mean() for g in groups})}).reindex(pnl.index)
d5.index.name = 'component'
d5_show = pd.DataFrame({'Contribution (pp)': [f'{100 * v:+.1f}' for v in d5.contribution],
                        'Months long': ['' if np.isnan(v) else f'{v:.0%}' for v in d5.months_long_share],
                        'Months short': ['' if np.isnan(v) else f'{v:.0%}' for v in d5.months_short_share]}, index=d5.index)
G.write_table(d5, 'd5_industry_pnl', label=P.POSTHOC, display=d5_show, align='lrrr')
G.industry_pnl(pnl)
register('D5', 'A3 test-period P&L by industry (sum of monthly contributions)', P.POSTHOC, 'd5_industry_pnl',
         'sealed/initial/ASOF_A3_weights.csv, ASOF_A3_ledger.csv; analysis/data/french', d5_show,
         f'Components add up to the sum of A3 monthly net returns ({100 * l3.net.sum():+.1f} pp).')
d5.round(4)

,contribution,months_long_share,months_short_share
component,,,
Mining & logging,0.0721,0.3310,0.3592
Construction,-0.1918,0.3592,0.3873
Durable mfg,0.0176,0.3099,0.2394
Nondurable mfg,0.0542,0.3239,0.3451
Wholesale,0.1491,0.3099,0.4225
Retail,0.0023,0.3873,0.3451
Transport & utilities,0.0143,0.4718,0.2958
Information,-0.0583,0.3028,0.3732
Finance & insurance,-0.1630,0.3380,0.4507


---
##### D6 Factor loadings
CONFIRMATORY factor regressions of the preregistered books, read from the frozen results.json.

In [13]:
FACTORS = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'Mom', 'group_momentum']
load_names = {'A0': 'A0 fixed rule', 'A1': 'A1 ridge', 'A3': 'A3 primary'}
rows = []
for s, label in load_names.items():
    a = res['attribution'][s]
    for f in ['intercept'] + FACTORS:
        rows.append({'strategy': label, 'factor': f, 'coef': a['coefficients'][f], 'se': a['standard_errors'][f],
                     't': a['t_statistics'][f], 'n': a['n'], 'label': P.CONFIRMATORY})
d6 = pd.DataFrame(rows)
d6_show = pd.DataFrame({lab: [(f'{100 * c:+.2f}% ({t:+.2f})' if f == 'intercept' else f'{c:+.2f} ({t:+.2f})')
                              for f, c, t in d6.loc[d6.strategy == lab, ['factor', 'coef', 't']].itertuples(index=False)]
                        for lab in load_names.values()},
                       index=pd.Index(['Alpha (per month)'] + ['Market', 'SMB', 'HML', 'RMW', 'CMA', 'Momentum', 'Industry momentum'], name='Factor'))
G.write_table(d6.set_index(['strategy', 'factor']), 'd6_factor_loadings', label=P.CONFIRMATORY, display=d6_show, align='lrrr')
G.loadings(d6[d6.factor != 'intercept'])
register('D6', 'Factor loadings of net long-short returns, test period (t in brackets)', P.CONFIRMATORY, 'd6_factor_loadings',
         'study1_preregistered/.../sealed/initial/results.json -> attribution', d6_show,
         'Newey-West, 6 lags. Gate 1 re-estimated these from the saved ledgers and matched to < 1e-8.')
d6_show

,A0 fixed rule,A1 ridge,A3 primary
Factor,,,
Alpha (per month),-0.25% (-0.92),-0.35% (-1.72),-0.38% (-1.86)
Market,-0.01 (-0.31),-0.01 (-0.20),-0.03 (-0.43)
SMB,+0.02 (+0.21),+0.12 (+1.01),+0.10 (+0.89)
HML,+0.09 (+0.60),-0.15 (-1.97),-0.15 (-2.01)
RMW,+0.00 (+0.05),+0.21 (+2.33),+0.19 (+2.16)
CMA,-0.04 (-0.21),+0.05 (+0.47),+0.03 (+0.26)
Momentum,-0.06 (-0.72),+0.01 (+0.15),-0.01 (-0.17)
Industry momentum,+0.22 (+1.93),-0.02 (-0.21),-0.01 (-0.12)


---
##### D7 Agent features
Research IC vs test IC of every valid repetition-1 candidate in both studies; selected and regime-gated ones marked.

In [14]:
decode = {v: k for k, v in json.loads((P.SONNET / 'outputs/blind_map.json').read_text())['variables'].items()}
pretty = {'JOR': 'Openings', 'HIR': 'Hires', 'QUR': 'Quits', 'LDR': 'Layoffs', 'H': 'Hours', 'E': 'Earnings'}
rows = []
for study, root in [('Sonnet', P.SONNET), ('Opus', P.OPUS)]:
    sel = json.loads((root / 'outputs/selected_features.json').read_text())
    out = json.loads((root / 'outputs/sealed/initial/results.json').read_text())
    specs = {x['candidate_id']: x for x in sel['seed_one_specs']}
    chosen = {c for v in sel['selected'].values() for c in v}
    for a in out['candidate_accounting']:
        cid = a['candidate_id']
        if a.get('valid') is False:
            rows.append({'study': study, 'candidate': cid, 'valid': False})
            continue
        expr = specs[cid]['spec']['expression']
        decoded = expr
        for vid, name in decode.items():
            decoded = decoded.replace(vid, pretty[name])
        rows.append({'study': study, 'candidate': cid, 'valid': True, 'arm': specs[cid]['arm'], 'expression': expr,
                     'decoded': decoded, 'research_ic': a['research']['mean'], 'research_t': a['research']['t'],
                     'research_n': a['research']['n'], 'test_ic': a['sealed']['mean'], 'test_t': a['sealed']['t'],
                     'test_n': a['sealed']['n'], 'passed_research': a['passed_research_filter'],
                     'selected': cid in chosen, 'gated': 'regime(' in expr})
d7 = pd.DataFrame(rows).astype({'research_ic': float, 'test_ic': float})
valid = d7[d7.valid == True].copy()
G.agents_scatter(valid)
summary = {}
for study in ['Sonnet', 'Opus']:
    v = valid[valid.study == study]
    both = v.dropna(subset=['research_ic', 'test_ic'])
    summary[study] = {'valid candidates': len(v), 'regime-gated': int(v.gated.sum()),
                      'test IC undefined (<=3 active months)': int(v.test_ic.isna().sum()),
                      'corr(research IC, test IC)': float(both.research_ic.corr(both.test_ic)),
                      'passed research filter': int(v.passed_research.sum()), 'selected': int(v.selected.sum()),
                      'selected and regime-gated': int((v.selected & v.gated).sum()),
                      'selected with positive test IC': int((v.selected & (v.test_ic > 0)).sum())}
d7_summary = pd.DataFrame(summary)
d7_summary.index.name = 'item'
d7.to_csv(P.table_path('d7_agent_candidates'), index=False)
chosen_rows = valid[valid.selected].copy()
d7_show = pd.DataFrame({'Study': chosen_rows.study, 'Arm': chosen_rows.arm, 'Feature (decoded)': chosen_rows.decoded.str.replace(',', ', '),
                        'Research IC (n)': [f'{a:+.3f} ({int(n)})' for a, n in zip(chosen_rows.research_ic, chosen_rows.research_n)],
                        'Test IC (n)': [('n/a' if not np.isfinite(a) else f'{a:+.3f}') + f' ({int(n)})' for a, n in zip(chosen_rows.test_ic, chosen_rows.test_n)]})
d7_summary_show = d7_summary.apply(lambda row: [f'{v:+.2f}' if row.name.startswith('corr') else f'{int(v)}' for v in row], axis=1, result_type='broadcast').astype(str)
d7_summary_show.index = [i.replace('<=', '≤') for i in d7_summary_show.index]
d7_summary_show.index.name = 'Repetition-1 candidates'
G.write_table(d7_summary, 'd7_agent_summary', label=P.POSTHOC, display=d7_summary_show, align='lrr')
G.write_table(chosen_rows.set_index('candidate')[['study', 'arm', 'decoded', 'research_ic', 'research_n', 'test_ic', 'test_n', 'gated']],
              'd7_selected_features', label=P.POSTHOC, display=d7_show.set_index('Study'), align='llp{2.55in}rr')
register('D7a', 'Agent candidates, research vs test (repetition 1, both studies)', P.POSTHOC, 'd7_agent_summary',
         'sealed/initial/results.json -> candidate_accounting; selected_features.json (both studies)', d7_summary_show)
register('D7b', 'Selected agent features, decoded after evaluation', P.POSTHOC, 'd7_selected_features',
         'selected_features.json, results.json, blind_map.json (both studies)', d7_show.set_index('Study'),
         'Full candidate list: analysis/output/study1/tables/d7_agent_candidates.csv. Test IC is undefined when a feature is active in 3 or fewer months.')
d7_summary

,Sonnet,Opus
item,,
valid candidates,33.000000,35.000000
regime-gated,9.000000,6.000000
test IC undefined (<=3 active months),4.000000,2.000000
"corr(research IC, test IC)",0.224763,0.372355
passed research filter,4.000000,7.000000
selected,4.000000,4.000000
selected and regime-gated,2.000000,2.000000
selected with positive test IC,0.000000,2.000000


---
##### D8 A3 windows
CONFIRMATORY window statistics of the primary strategy from the frozen results.json.

In [15]:
win = res['windows']['A3']
labels8 = {'whole': 'Full test, Nov 2014-Aug 2026', 'first_half': 'First half', 'second_half': 'Second half',
           'ex_pandemic': 'Excluding Mar-Dec 2020', 'last_18': 'Last 18 months (Mar 2025-Aug 2026)'}
rows = {labels8[k]: {'months': win[k]['n'], 'sharpe': win[k]['sharpe'], 'net_ann': win[k]['mean_net_annual'],
                     'ic': win[k]['IC']['mean'], 'ic_t': win[k]['IC']['t'], 'scope': 'test'} for k in labels8}
for k, lab in [('full', 'Course window "full" (from Jun 2007)'), ('post_2010', 'Course window "post-2010"')]:
    c = res['course_windows']['A3'][k]
    rows[lab] = {'months': c['n'], 'sharpe': c['sharpe'], 'net_ann': c['mean_net_annual'], 'ic': c['IC']['mean'],
                 'ic_t': c['IC']['t'], 'scope': 'research (pseudo-real-time) + test, descriptive'}
d8 = pd.DataFrame(rows).T
d8.index.name = 'window'
d8_show = pd.DataFrame({'Months': d8.months.astype(int), 'Sharpe': [f'{v:+.2f}' for v in d8.sharpe],
                        'Net return p.a.': [f'{v:+.1%}' for v in d8.net_ann], 'Mean IC (t)': ic_text(d8.ic.astype(float), d8.ic_t.astype(float))},
                       index=d8.index)
G.write_table(d8, 'd8_a3_windows', label=P.CONFIRMATORY, display=d8_show, align='lrrrr')
register('D8', 'Primary A3 by window', P.CONFIRMATORY, 'd8_a3_windows',
         'study1_preregistered/.../sealed/initial/results.json -> windows.A3, course_windows.A3', d8_show,
         'Course windows mix the pseudo-real-time research period with the test period; the frozen study labels them descriptive.')
d8_show

,Months,Sharpe,Net return p.a.,Mean IC (t)
window,,,,
"Full test, Nov 2014-Aug 2026",142,-0.58,-4.6%,-0.021 (-1.08)
First half,71,-0.72,-4.9%,-0.028 (-0.90)
Second half,71,-0.49,-4.3%,-0.015 (-0.59)
Excluding Mar-Dec 2020,132,-0.51,-4.1%,-0.012 (-0.56)
Last 18 months (Mar 2025-Aug 2026),18,+0.85,+6.7%,+0.033 (+0.44)
"Course window ""full"" (from Jun 2007)",231,-0.44,-3.2%,-0.009 (-0.49)
"Course window ""post-2010""",200,-0.62,-4.4%,-0.019 (-1.02)


---
##### D9 Risk overlay
How often the gross cap bound and how ex-ante volatility compares with realised volatility, from the saved test ledgers.

In [16]:
rows = []
for study, root, strategies in [('Sonnet', P.SONNET, ['A0', 'A1', 'A1-T', 'A3', 'A4']), ('Opus', P.OPUS, ['A3', 'A4'])]:
    for s in strategies:
        l = D.saved_csv(f'ASOF_{s}_ledger.csv', root)
        rows.append({'study': study, 'strategy': s, 'months': len(l),
                     'cap_binding_share': float((l.gross_leverage >= P.PORTFOLIO['gross_cap'] - 1e-3).mean()),
                     'mean_ex_ante_vol': float(l.ex_ante_vol.mean()), 'realised_vol': float(l.net.std() * np.sqrt(12)),
                     'mean_gross_leverage': float(l.gross_leverage.mean()), 'mean_abs_hedge': float(l.market_hedge.abs().mean())})
d9 = pd.DataFrame(rows).set_index(['study', 'strategy'])
d9_show = pd.DataFrame({'Cap binds': [f'{v:.0%}' for v in d9.cap_binding_share], 'Ex-ante vol': [f'{v:.1%}' for v in d9.mean_ex_ante_vol],
                        'Realised vol': [f'{v:.1%}' for v in d9.realised_vol], 'Gross leverage': [f'{v:.2f}' for v in d9.mean_gross_leverage],
                        '|Hedge|': [f'{v:.2f}' for v in d9.mean_abs_hedge]},
                       index=pd.Index([f'{a} {b}' for a, b in d9.index], name='Study, strategy'))
G.write_table(d9, 'd9_risk_overlay', label=P.POSTHOC, display=d9_show, align='lrrrrr')
register('D9', 'Risk overlay in the test period (target volatility 10%, gross cap 2)', P.POSTHOC, 'd9_risk_overlay',
         'sealed/initial/ASOF_*_ledger.csv (both studies)', d9_show,
         'Cap binds = share of months with gross leverage >= 1.999. Opus A0/A1/A1-T are identical to Sonnet and omitted.')
d9_show

,Cap binds,Ex-ante vol,Realised vol,Gross leverage,|Hedge|
"Study, strategy",,,,,
Sonnet A0,80%,8.0%,9.7%,1.96,0.15
Sonnet A1,88%,7.5%,7.9%,1.98,0.13
Sonnet A1-T,82%,7.5%,8.2%,1.97,0.12
Sonnet A3,87%,7.5%,7.8%,1.98,0.13
Sonnet A4,91%,7.3%,7.8%,1.99,0.13
Opus A3,92%,7.3%,7.5%,1.99,0.15
Opus A4,93%,7.3%,8.4%,1.99,0.11


---
##### D10 Agent experiment design
Diagram and parameter table of the blinded agent experiment, read from Study 1.

In [17]:
G.agents_diagram()
design_rows = []
for study, root in [('Sonnet', P.SONNET), ('Opus', P.OPUS)]:
    cand = pd.read_csv(root / 'outputs/candidates.csv')
    mig = json.loads((root / 'outputs/agents/migration_summary.json').read_text())
    hr = json.loads((root / 'outputs/human_review_submission.json').read_text())
    items = hr['items']
    design_rows += [
        (study, 'Valid proposals', f"{int(cand.valid.sum())} / {len(cand)}", 'outputs/candidates.csv'),
        (study, 'Migration reports admitted', f"{mig['reports_admitted']} / {mig['reports_total']}", 'outputs/agents/migration_summary.json'),
        (study, 'Defined LLM-judge uptake labels', f"{mig['defined_judgments']} / {mig['n']}", 'outputs/agents/migration_summary.json'),
        (study, 'Human-audited judge pairs', f"{len(items)}", 'outputs/human_review_submission.json')]
    if 'agreement_count' in hr:
        design_rows.append((study, 'Human-judge agreement', f"{hr['agreement_count']} / {hr['reviewed']} (human yes {hr['human_yes']}, judge yes {hr['machine_yes_in_sample']})",
                            'outputs/human_review_submission.json'))
A = FP.AGENT
design_rows = [('Both', 'Arms', 'islands (communicating), independent (control)', 'params.py AGENT'),
               ('Both', 'Agents per arm', ', '.join(A['emphases']), 'params.py AGENT'),
               ('Both', 'Proposals per agent x repetitions', f"{A['submissions']} x {len(A['seeds'])}", 'params.py AGENT'),
               ('Both', 'Migration reports', f"after proposals {' and '.join(map(str, A['migration_after']))}, at most {A['migration_word_limit']} words", 'params.py AGENT'),
               ('Both', 'Selection (repetition 1 only)', f"t ≥ {FP.MODEL['candidate_min_t']}, same sign in both halves, |corr| < {FP.MODEL['candidate_max_correlation']}, at most {FP.MODEL['candidate_max']} per arm", 'params.py MODEL')] + design_rows
d10 = pd.DataFrame(design_rows, columns=['study', 'item', 'value', 'source'])
G.write_table(d10, 'd10_agent_design', index=False, label=P.CONFIRMATORY, display=d10[['study', 'item', 'value']], align='llp{3.3in}')
register('D10', 'Agent experiment design and process counts', P.CONFIRMATORY, 'd10_agent_design',
         'frozen params.py, candidates.csv, migration_summary.json, human_review_submission.json (both studies)', d10.set_index('study'))
d10

,study,item,value,source
0,Both,Arms,"islands (communicating), independent (control)",params.py AGENT
1,Both,Agents per arm,"Temporal, Composition, Interactions",params.py AGENT
2,Both,Proposals per agent x repetitions,6 x 3,params.py AGENT
3,Both,Migration reports,"after proposals 2 and 4, at most 120 words",params.py AGENT
4,Both,Selection (repetition 1 only),"t ≥ 1.5, same sign in both halves, |corr| < 0....",params.py MODEL
5,Sonnet,Valid proposals,98 / 108,outputs/candidates.csv
6,Sonnet,Migration reports admitted,7 / 18,outputs/agents/migration_summary.json
7,Sonnet,Defined LLM-judge uptake labels,8 / 36,outputs/agents/migration_summary.json
8,Sonnet,Human-audited judge pairs,8,outputs/human_review_submission.json
9,Opus,Valid proposals,106 / 108,outputs/candidates.csv


---
##### Diagnostics report
Write analysis/output/study1/diagnostics_v2.md with every number and the file it came from.

In [18]:
figures = sorted(p.name for p in P.FIGS.glob('fig_*.pdf'))
parts = [f'# Diagnostics v2 (Part 2)\n\nGenerated by analysis/run.ipynb on {utc_now()}. Every number below is copied from the CSV '
         'named in its header (analysis/output/study1/tables/), which the notebook writes from the sources listed. Labels: CONFIRMATORY, POST-HOC or FORWARD TEST. '
         'Historical labor signals use revised FRED data with fixed lags (see data_vintage.md). Figures (vector PDF): '
         + ', '.join(f'`report/figures/{f}`' for f in figures) + '.\n']
for r in REGISTRY:
    parts += [f"## {r['item']}. {r['title']}\n", f"Label: **{r['label']}**. File: `analysis/output/study1/tables/{r['name']}.csv` "
              f"(LaTeX: `report/tables/{r['name']}.tex`). Source: {r['sources']}.\n", G.markdown(r['df']), '']
    if r['note']:
        parts += [r['note'], '']
(P.OUT1 / 'diagnostics_v2.md').write_text('\n'.join(parts))
print('figures:', figures)

figures: ['fig_agents.pdf', 'fig_agents_scatter.pdf', 'fig_cumulative.pdf', 'fig_horizon.pdf', 'fig_horizon_profile.pdf', 'fig_ic_signals.pdf', 'fig_industry_pnl.pdf', 'fig_loadings.pdf', 'fig_s1_horizon.pdf', 'fig_s23_agents_seeds.pdf', 'fig_s23_gross_needed.pdf', 'fig_tightness.pdf', 'fig_timeline.pdf', 'fig_v2_windows.pdf', 'fig_v2d_dropone.pdf', 'fig_v2d_holding.pdf', 'fig_v2d_tightness.pdf']


---
##### Study 2.1 freeze check
The Study 2.1 specification must be unchanged since it was hashed and committed (44ae263) before any Study 2.1 result existed.

In [19]:
spec_bytes = (P.SPECS / 'study2_1_variants.md').read_bytes()
spec_record = (P.SPECS / 'study2_1_variants.sha256').read_text().split()
SPEC_HASH, SPEC_UTC = spec_record[0], spec_record[3]
if hashlib.sha256(spec_bytes).hexdigest() != SPEC_HASH:
    stop('analysis/specs/study2_1_variants.md differs from the hash recorded in analysis/specs/study2_1_variants.sha256')
print('v2 spec frozen at', SPEC_UTC, 'sha256', SPEC_HASH[:16])

v2 spec frozen at 2026-10-03T09:00:13+00:00 sha256 ca6582347904489b


---
##### Study 2.1 variants
Scores, one continuous backtest per variant and cost level (May 2004-Aug 2026), exactly as frozen in v2_spec.md. POST-HOC.

In [20]:
Z = {k: F[k] for k in ['F1', 'F2', 'F3', 'F4', 'F5', 'W']}
res_months = months('research')
V2B_RES_IC = {k: float(S.rank_ic(Z[k].reindex(res_months), R['relative'].reindex(res_months)).mean()) for k in Z}
V2B_SIGNS = {k: (1.0 if v >= 0 else -1.0) for k, v in V2B_RES_IC.items()}
V2 = {
    'V2a': (D.csrank(Z['W']), 3),
    'V2b': (sum(V2B_SIGNS[k] * Z[k] for k in Z) / 6, 3),
    'V2c': ((D.csrank(Z['W']) + D.csrank(F['IndMom'])) / 2, 3),
    'V2d': (-(Z['F2'] + Z['F3']) / 2, 12),
}
full_months = months('full')
LEDGERS, WEIGHTS = {}, {}
for v, (score, holding) in V2.items():
    for cost in P.COST_GRID_BPS:
        ledger, weights = D.backtest(score.reindex(full_months), R, holding=holding, cost_bps=cost)
        LEDGERS[(v, cost)] = ledger
        if cost == 10:
            WEIGHTS[v] = weights
print('backtests done:', len(LEDGERS))

backtests done: 12


In [21]:
WINDOW_NAMES = list(P.WINDOWS)
rows, readings = [], []
for v, (score, holding) in V2.items():
    h = 3 if holding == 3 else 12
    target_h = D.horizon_target(R['total'], h)
    for w in WINDOW_NAMES:
        idx = months(w)
        l10 = LEDGERS[(v, 10)].loc[idx]
        pf = S.perf(l10)
        ic_series = S.rank_ic(score.reindex(idx), R['relative'].reindex(idx))
        ic1 = S.hac(ic_series, lags=P.HAC_LAGS)
        ich = S.hac(S.rank_ic(score.reindex(idx), target_h.reindex(idx)), lags=h + 5)
        fa = S.factor_attribution(l10.net, R)
        placebo = S.placebo(score, R['relative'], idx) if len(idx) >= 2 * P.PLACEBO_MIN_SHIFT + 1 else {'p': np.nan, 'n_shifts': 0}
        dsr = S.deflated_sharpe(l10.net, n_trials=P.N_TRIALS)
        half = len(idx) // 2
        row = {'variant': v, 'window': w, 'months': pf['n'], 'sharpe': pf['sharpe'], 'net_ann': pf['net_ann'], 'vol': pf['vol'],
               'max_dd': pf['max_dd'], 'hit': pf['hit'], 'ic': ic1['mean'], 'ic_t': ic1['t'], 'ic_h': h, 'ic_hold': ich['mean'],
               'ic_hold_t': ich['t'], 'alpha_month': fa['mean'], 'alpha_t': fa['t'],
               'ic_first_half': ic_series.iloc[:half].mean(), 'ic_second_half': ic_series.iloc[half:].mean(),
               'turnover': pf['turnover'], 'cost_ann': pf['cost_ann'], 'placebo_p': placebo['p'], 'placebo_shifts': placebo['n_shifts'],
               'dsr': dsr['dsr'], 'dsr_benchmark_annual_sharpe': dsr['benchmark_annual_sharpe']}
        for cost in P.COST_GRID_BPS:
            row[f'sharpe_{cost}bp'] = S.perf(LEDGERS[(v, cost)].loc[idx])['sharpe']
        rows.append(row)
V2_STATS = pd.DataFrame(rows)
V2_STATS.to_csv(P.table_path('v2_window_stats'), index=False)

# Reading rule (v2_spec.md section 6), test window only.
test_rows = V2_STATS[V2_STATS.window == 'test'].set_index('variant')
criteria = pd.DataFrame({
    'sharpe_25bp_positive': test_rows['sharpe_25bp'] > 0,
    'ic_t_ge_2': test_rows['ic_t'] >= 2.0,
    'alpha_t_ge_2': test_rows['alpha_t'] >= 2.0,
    'ic_positive_both_halves': (test_rows['ic_first_half'] > 0) & (test_rows['ic_second_half'] > 0),
    'placebo_p_lt_0.05': test_rows['placebo_p'] < 0.05,
    'dsr_gt_0.95': test_rows['dsr'] > 0.95})
criteria['reading'] = np.where(criteria.all(axis=1), 'consistent with a real effect, to be forward-tested',
                               'not distinguishable from noise after the looks already taken')
criteria.to_csv(P.table_path('v2_reading_rule'))

# V2b signs, monthly net returns and P&L by industry.
pd.DataFrame({'research_ic': V2B_RES_IC, 'sign': V2B_SIGNS}).rename_axis('signal').to_csv(P.table_path('v2b_signs'))
pd.DataFrame({f'{v}_{c}bp': l.net for (v, c), l in LEDGERS.items()}).to_csv(P.table_path('v2_monthly_net'))
pnl_rows = []
for v in V2:
    w_v, l_v = WEIGHTS[v], LEDGERS[(v, 10)]
    for win in ['test', 'full']:
        idx = months(win)
        contrib = (w_v.loc[idx, list(P.GROUPS)] * R['excess'].loc[idx, list(P.GROUPS)]).sum().rename(index=P.NAMES)
        parts = pd.concat([contrib, pd.Series({'Market hedge': float((w_v.loc[idx, 'market'] * R['factors'].loc[idx, 'Mkt-RF']).sum()),
                                               'Trading costs': -float(l_v.loc[idx, 'cost'].sum() + l_v.loc[idx, 'borrow_financing'].sum())})])
        assert abs(parts.sum() - l_v.loc[idx, 'net'].sum()) < 1e-10
        pnl_rows += [{'variant': v, 'window': win, 'component': k, 'contribution': float(val)} for k, val in parts.items()]
V2_PNL = pd.DataFrame(pnl_rows)
V2_PNL.to_csv(P.table_path('v2_pnl_by_industry'), index=False)
# The variants were run once (v2_run_log.md). A later execution must reproduce that run byte for byte.
run_log = P.OUT2 / 'v2_run_log.md'
REGENERATED_UTC = utc_now()
if run_log.exists():
    log_text = run_log.read_text()
    for name, digest in re.findall(r'\| `(tables/[^`]+)` \| `([0-9a-f]{64})` \|', log_text):
        if sha256(P.OUT2 / name) != digest:
            stop(f'v2 output {name} differs from the first run recorded in v2_run_log.md; v2 results must not change')
    RESULTS_UTC = re.search(r'FIRST_RUN_UTC (\S+)', log_text).group(1)
else:
    RESULTS_UTC = REGENERATED_UTC
print('v2 first run', RESULTS_UTC, '| this execution', REGENERATED_UTC)

v2 first run 2026-10-03T09:01:55+00:00 | this execution 2026-10-04T10:22:17+00:00


---
##### Study 2.1 results
Write v2_results.md and the LaTeX summary, and add W to the cumulative and loadings figures. Every variant is reported.

In [22]:
# A3 contrast (CONFIRMATORY, frozen files) for fig_v2_windows.
rs = json.loads((P.SONNET / 'outputs/research_summary.json').read_text())
A3_CONTRAST = {'research': rs['strategies']['A3']['sharpe'], 'test': res['windows']['A3']['whole']['sharpe'],
               'full': res['course_windows']['A3']['full']['sharpe'], 'post-2010': res['course_windows']['A3']['post_2010']['sharpe'],
               'last 18m': res['windows']['A3']['last_18']['sharpe']}
sharpe_grid = V2_STATS.pivot(index='variant', columns='window', values='sharpe')[WINDOW_NAMES]
G.v2_windows(sharpe_grid, pd.Series(A3_CONTRAST))

# Compact LaTeX summary.
show = pd.DataFrame(index=pd.Index(list(V2), name='Variant'))
for w in WINDOW_NAMES:
    show[{'research': 'Research', 'test': 'Test', 'full': 'Full', 'post-2010': 'Post-2010', 'last 18m': 'Last 18m'}[w]] = [f'{v:+.2f}' for v in sharpe_grid[w]]
show['Test IC (t)'] = [f'{a:+.3f} ({b:+.2f})' for a, b in zip(test_rows.ic, test_rows.ic_t)]
show['Alpha t'] = [f'{v:+.2f}' for v in test_rows.alpha_t]
show['Placebo p'] = [f'{v:.2f}' for v in test_rows.placebo_p]
show['DSR'] = [f'{v:.2f}' for v in test_rows.dsr]
G.write_table(sharpe_grid, 'v2_results', label=P.POSTHOC, display=show, align='l' + 'r' * 9)

# V2a added to the cumulative and loadings figures (test window, 10 bp).
v2a_test = LEDGERS[('V2a', 10)].loc[months('test'), 'net']
G.cumulative({names[s]: l.net for s, l in conf_ledgers.items()},
             {'Industry momentum 12-1 (post-hoc)': (mom_ledger.net.loc[months('test')], ':'),
              'V2a wage growth (post-hoc)': (v2a_test, '--')})
fa_v2a = S.factor_attribution(v2a_test, R)
v2a_load = pd.DataFrame([{'strategy': 'V2a wage growth', 'factor': f, 'coef': fa_v2a['coef'][f], 'se': fa_v2a['se_all'][f],
                          't': fa_v2a['tstat'][f], 'n': fa_v2a['n'], 'label': P.POSTHOC} for f in ['intercept'] + FACTORS])
G.loadings(pd.concat([d6, v2a_load])[lambda x: x.factor != 'intercept'])
v2a_load.to_csv(P.table_path('v2a_factor_loadings_test'), index=False)
print('figures and LaTeX written')

figures and LaTeX written


In [23]:
lines = [f'# v2 results (POST-HOC)\n',
         f'First run at **{RESULTS_UTC}** (this file regenerated at {REGENERATED_UTC}; outputs verified identical to the first run, see v2_run_log.md). '
         f'Specification `analysis/specs/study2_1_variants.md` was frozen at **{SPEC_UTC}** '
         f'(sha256 `{SPEC_HASH}`, commit 44ae263) before this run. All four variants are reported. Labor data are revised FRED values '
         'with fixed lags (`data_vintage.md`); returns and portfolio construction are the frozen study\'s (Gate 1).\n',
         'The confirmatory verdict is unchanged: primary A3, test Sharpe ' + f"{res['windows']['A3']['whole']['sharpe']:+.2f}" + ', **Do not implement**.\n',
         '## Reading rule (v2_spec.md section 6), test window\n',
         'Source: `analysis/output/study2/tables/v2_reading_rule.csv`.\n', G.markdown(criteria), '',
         '## Sharpe ratio by window (10 bp)\n', 'Source: `analysis/output/study2/tables/v2_window_stats.csv` (column `sharpe`).\n',
         G.markdown(sharpe_grid.round(3)), '',
         '## A3 contrast (CONFIRMATORY)\n', 'Sources: research_summary.json (research, 88 months Jun 2007-Sep 2014); results.json windows (test, last 18m) '
         'and course_windows (full, post-2010; descriptive mix of research and test).\n', G.markdown(pd.Series(A3_CONTRAST, name='A3 Sharpe').to_frame().round(3)), '',
         '## All statistics, every variant and window\n', 'Source: `analysis/output/study2/tables/v2_window_stats.csv`. Net return, volatility, drawdown and hit rate in decimals; '
         'alpha per month; DSR uses N_trials = 112 and trial variance 1/T.\n']
for v in V2:
    sub = V2_STATS[V2_STATS.variant == v].set_index('window')
    cols = ['months', 'sharpe', 'net_ann', 'vol', 'max_dd', 'hit', 'ic', 'ic_t', 'ic_hold', 'ic_hold_t', 'alpha_month', 'alpha_t',
            'sharpe_0bp', 'sharpe_10bp', 'sharpe_25bp', 'turnover', 'placebo_p', 'dsr']
    lines += [f'### {v}: {P.VARIANTS[v]}\n', G.markdown(sub[cols].T.round(4)), '']
lines += ['## V2b signs, learned on research months only (May 2004-Sep 2014)\n', 'Source: `analysis/output/study2/tables/v2b_signs.csv`.\n',
          G.markdown(pd.DataFrame({'research_ic': V2B_RES_IC, 'sign': V2B_SIGNS}).round(4)), '',
          '## P&L by industry (sum of monthly contributions, 10 bp)\n', 'Source: `analysis/output/study2/tables/v2_pnl_by_industry.csv`.\n',
          G.markdown(V2_PNL.assign(col=V2_PNL.variant + ' ' + V2_PNL.window).pivot_table(index='component', columns='col', values='contribution', sort=False).round(3)), '',
          '## Files\n', '- `analysis/output/study2/tables/v2_window_stats.csv`, `v2_reading_rule.csv`, `v2b_signs.csv`, `v2_monthly_net.csv`, '
          '`v2_pnl_by_industry.csv`, `v2a_factor_loadings_test.csv`', '- `report/tables/v2_results.tex`',
          '- `report/figures/fig_v2_windows.pdf`; V2a added to `fig_cumulative.pdf` and `fig_loadings.pdf`', '']
(P.OUT2 / 'v2_results.md').write_text('\n'.join(lines))
print(criteria.to_string())
sharpe_grid.round(2)

         sharpe_25bp_positive  ic_t_ge_2  alpha_t_ge_2  ic_positive_both_halves  placebo_p_lt_0.05  dsr_gt_0.95                                                       reading
variant                                                                                                                                                                      
V2a                      True       True         False                     True               True        False  not distinguishable from noise after the looks already taken
V2b                     False      False         False                     True              False        False  not distinguishable from noise after the looks already taken
V2c                      True      False         False                     True              False        False  not distinguishable from noise after the looks already taken
V2d                      True      False         False                    False              False        False  not distinguishab

window,research,test,full,post-2010,last 18m
variant,,,,,
V2a,0.09,0.16,0.15,0.12,-1.71
V2b,-0.06,-0.15,-0.11,-0.16,-1.34
V2c,0.27,0.35,0.32,0.24,-0.78
V2d,0.50,0.29,0.35,0.30,0.29


---
##### HQ12 12-month placebo
Additional check added after the run (POST-HOC, not part of the reading rule): circular-shift placebo on 12-month IC to match HQ12's holding period.

In [24]:
target_12 = D.horizon_target(R['total'], 12)
rows = []
for w in ['research', 'test', 'full', 'post-2010']:
    pl = S.placebo(V2['V2d'][0], target_12, months(w))
    rows.append({'window': w, 'ic_12m': pl['actual_ic'], 'placebo_p_12m': pl['p'], 'shifts': pl['n_shifts'],
                 'placebo_p_1m_predeclared': float(V2_STATS.set_index(['variant', 'window']).loc[('V2d', w), 'placebo_p'])})
V2D_PLACEBO_12 = pd.DataFrame(rows).set_index('window')
V2D_PLACEBO_12.to_csv(P.table_path('v2d_placebo_12m'))
section_text = ['', '## Additional check added after the run (POST-HOC, not part of the reading rule)', '',
                'Added on 3 Oct 2026 after the v2 results above had been seen, at the team\'s request. It does **not** replace the '
                'pre-declared placebo: V2d\'s pre-declared test-window placebo p (1-month IC) stays '
                f"{V2D_PLACEBO_12.loc['test', 'placebo_p_1m_predeclared']:.2f}, as reported in the tables above and in the reading rule.", '',
                'V2d is held 12 months, so this check uses its 12-month IC: the rank IC between the score and the 12-month compounded '
                'relative return (months m to m+11). The score panel is rolled circularly against that target within each window by every shift '
                'from 24 to n-24 months; p is the share of shifted mean ICs at or above the actual one. Months at the end of the sample without a '
                'complete 12-month return drop out. The 12-month returns overlap, which the circular shift preserves but does not correct.', '',
                'Source: `analysis/output/study2/tables/v2d_placebo_12m.csv`.', '', G.markdown(V2D_PLACEBO_12.round(4)), '']
with open(P.OUT2 / 'v2_results.md', 'a') as handle:
    handle.write('\n'.join(section_text))
V2D_PLACEBO_12

,ic_12m,placebo_p_12m,shifts,placebo_p_1m_predeclared
window,,,,
research,0.047077,0.256410,78,0.602564
test,0.096175,0.063158,95,0.852632
full,0.071129,0.036199,221,0.855204
post-2010,0.080673,0.000000,153,0.797386


---
##### Forward test: data
ALFRED values as published on 29 Sep 2026; per amendment 001 the run stops only if an observation later than the fixed-lag month is used. FORWARD TEST.

In [25]:
import shutil
import tempfile
from pathlib import Path
fwd_record = (P.SPECS / 'forward_spec.sha256').read_text().split()
FWD_HASH, FWD_UTC = fwd_record[0], fwd_record[3]
amd_record = (P.SPECS / 'forward_amendment_001.sha256').read_text().split()
AMD_HASH, AMD_UTC = amd_record[0], amd_record[3]
if sha256(P.SPECS / 'forward_spec.md') != FWD_HASH:
    stop('analysis/specs/forward_spec.md differs from the hash recorded when it was frozen')
if sha256(P.SPECS / 'forward_amendment_001.md') != AMD_HASH:
    stop('analysis/specs/forward_amendment_001.md differs from the hash recorded when it was frozen')
D.download_alfred()                                   # skips files already downloaded
USED = {'JOLTS': pd.Period('2026-07', freq='M'), 'CES': pd.Period('2026-08', freq='M')}   # fixed-lag months at the 30 Sep decision
family = lambda sid: 'JOLTS' if sid.startswith('JTU') or sid == 'JTSJOL' else 'CES'           # UNEMPLOY follows CES (lag 1)
carried = pd.DataFrame({sid: {'family': family(sid), 'last_carried': str(D.read_fred(sid, P.FRED_VINTAGE).index.max())} for sid in D.series_ids()}).T
print('vintage', P.VINTAGE_DATE, 'carries:', carried.groupby('family').last_carried.unique().to_dict())
# Every series must contain its fixed-lag month (otherwise the signal would silently be stale).
missing = [sid for sid in D.series_ids() if USED[family(sid)] not in D.read_fred(sid, P.FRED_VINTAGE).index]
if missing:
    stop(f'Vintage {P.VINTAGE_DATE} lacks the fixed-lag month for: {missing}')
# A copy of the vintage truncated at the fixed-lag months: scores must not change (nothing later is used).
truncated_dir = Path(tempfile.mkdtemp(prefix='alfred_truncated_'))
for sid in D.series_ids():
    frame = pd.read_csv(P.FRED_VINTAGE / f'{sid}.csv')
    keep = pd.PeriodIndex(pd.to_datetime(frame.iloc[:, 0]), freq='M') <= USED[family(sid)]
    frame[keep].to_csv(truncated_dir / f'{sid}.csv', index=False)

vintage 2026-09-29 carries: {'CES': array(['2026-08'], dtype=object), 'JOLTS': array(['2026-08'], dtype=object)}


---
##### Forward test: Q4 2026 book
W (headline), SC (secondary), W+MOM and HQ12 (amendment 001) tranches at the 30 Sep 2026 decision, with the frozen beta hedge and volatility scaling.

In [26]:
m_fwd = pd.Period(P.FORWARD_DECISION, freq='M') + 1                     # earning month Oct 2026
mom_window = pd.period_range(m_fwd - 12, m_fwd - 2, freq='M')            # Oct 2025 .. Aug 2026
assert len(mom_window) == 11 and R['total'].index.max() >= mom_window[-1]
mom_formula = lambda m: (1 + R['total'].loc[m - 12:m - 2]).prod() - 1
assert np.allclose(mom_formula(pd.Period('2026-08', freq='M')), F['IndMom'].loc['2026-08'])   # same definition as the backtests
MOM_FWD = mom_formula(m_fwd)


def forward_scores(folder):
    Fv, _ = D.signals(folder, last_decision=P.FORWARD_DECISION)
    Zv = {k: Fv[k].loc[[m_fwd]] for k in ['F1', 'F2', 'F3', 'F4', 'F5', 'W']}
    mom = pd.DataFrame([MOM_FWD.to_numpy()], index=pd.PeriodIndex([m_fwd], freq='M'), columns=MOM_FWD.index)
    return {'V2a': D.csrank(Zv['W']).loc[m_fwd],
            'V2b': (sum(V2B_SIGNS[k] * Zv[k] for k in Zv) / 6).loc[m_fwd],
            'V2c': ((D.csrank(Zv['W']) + D.csrank(mom)) / 2).loc[m_fwd],
            'V2d': (-(Zv['F2'] + Zv['F3']) / 2).loc[m_fwd]}


scores_fwd = forward_scores(P.FRED_VINTAGE)
scores_truncated = forward_scores(truncated_dir)
shutil.rmtree(truncated_dir)
for v, s in scores_fwd.items():
    if not np.isfinite(s).all() or s.std() == 0:
        stop(f'{v} forward score is missing or constant at the 30 Sep 2026 decision')
    if not np.array_equal(s.to_numpy(), scores_truncated[v].to_numpy()):
        stop(f'{v}: the forward score uses an observation later than the fixed-lag month (amendment 001 stop condition)')
print('fixed-lag check passed: scores from the full and the truncated vintage are identical for', list(scores_fwd))


def tranche(score):
    """Frozen backtest rule: stable sort by group ID, then by score; +1/3 top three, -1/3 bottom three."""
    rank = score.sort_index().sort_values(kind='stable')
    w = pd.Series(0.0, index=score.index)
    w[rank.index[-P.PORTFOLIO['longs']:]] = 1 / P.PORTFOLIO['longs']
    w[rank.index[:P.PORTFOLIO['shorts']]] = -1 / P.PORTFOLIO['shorts']
    return w


from sklearn.covariance import LedoitWolf
hist = pd.period_range(m_fwd - P.PORTFOLIO['risk_window'] - 1, m_fwd - 2, freq='M')     # 60 months ending Aug 2026
history = R['excess'].reindex(hist).join(R['factors']['Mkt-RF'].reindex(hist)).dropna()
assert len(history) >= P.PORTFOLIO['risk_min_months'] and history.index.max() == pd.Period('2026-08', freq='M')
market = history['Mkt-RF'].to_numpy()
BETAS = pd.Series({g: np.cov(history[g], market, ddof=1)[0, 1] / np.var(market, ddof=1) for g in P.GROUPS})
cov = LedoitWolf().fit(history.to_numpy()).covariance_
TRANCHES, HEDGE = {}, {}
for v, s in scores_fwd.items():
    w = tranche(s)
    hedge = -float(w @ BETAS)
    raw = np.r_[w.to_numpy(), hedge]
    risk = float(np.sqrt(raw @ cov @ raw * 12))
    scale = min(P.PORTFOLIO['vol_target'] / risk, P.PORTFOLIO['gross_cap'] / np.abs(raw).sum())
    TRANCHES[v] = w
    HEDGE[v] = {'variant': v, 'spy_hedge_weight_per_unit': hedge, 'ex_ante_vol_unit': risk, 'frozen_scale_factor': scale,
                'scaled_gross': scale * np.abs(raw).sum(), 'scaled_ex_ante_vol': scale * risk}
pd.DataFrame(HEDGE.values()).to_csv(P.FORWARD / 'book_2026Q4_hedge.csv', index=False)
BOOK_UTC = utc_now()   # replaced below by the first-build time if the book is unchanged
pd.DataFrame({**{f'{v}_score': s for v, s in scores_fwd.items()}, 'mom_12_1': MOM_FWD, 'beta_60m': BETAS}).rename_axis('group').to_csv(P.FORWARD / 'book_2026Q4_scores.csv')
print('book built at', BOOK_UTC)

fixed-lag check passed: scores from the full and the truncated vintage are identical for ['V2a', 'V2b', 'V2c', 'V2d']
book built at 2026-10-04T10:22:19+00:00


---
##### Forward test: ETF proxies
One liquid ETF per group from Study 1's issuer-verified list, the four books by group, and the netted ETF book. The weekly tracker is deferred (amendment 001).

In [27]:
comp = R['cap49'].loc['2026-08']
mix = {g: ', '.join(f'{k} {v:.0%}' for k, v in (comp[members.split()] / comp[members.split()].sum()).sort_values(ascending=False).head(3).items())
       for g, (_, _, _, members) in P.GROUPS.items()}
ETF = {  # group: (ETF, alternative, caveat). Source: frozen implementation_capacity.md (issuer pages checked 28 Sep 2026); SPY added for the hedge.
    1: ('XOP', 'XME', 'Group is mostly the French Oil portfolio, incl. integrated majors and refiners; XOP is equal-weight E&P. Metals and gold not covered; no logging exposure.'),
    2: ('ITB', 'XHB', 'Residential builders and suppliers; French Cnstr also holds contractors.'),
    3: ('XLI', '', 'Poor match: the group is about 60% semiconductors by market cap (Chips); XLI holds industrials. Shares XLI with groups 5 and 11.'),
    4: ('XLP', '', 'Poor match: the group is about 60% pharmaceuticals (Drugs), which XLP excludes.'),
    5: ('XLI', '', 'No wholesale ETF verified; XLI holds some distributors only. Shares XLI with groups 3 and 11.'),
    6: ('XRT', '', 'Equal-weight retail index; French Rtail is value-weighted.'),
    7: ('XLU', 'IYT', 'Group is about 61% utilities and 39% transport by cap; XLU covers utilities only.'),
    8: ('VOX', '', 'GICS communication services includes internet and media; French Telcm is telecom and cable.'),
    9: ('XLF', 'KBE; KIE', 'Sector index; does not replicate Banks, Insur and Fin weights.'),
    10: ('IYR', '', 'Mostly REITs, which French classifies outside RlEst (real estate operators).'),
    11: ('XLI', '', 'No professional-services ETF verified; XLI holds a small services slice. Shares XLI with groups 3 and 5.'),
    12: ('IHF', '', 'Includes health insurers; social-assistance coverage incomplete.'),
    13: ('PEJ', '', 'Broader leisure and entertainment exposure.'),
}
etf = pd.DataFrame([{'group': g, 'name': P.NAMES[g], 'etf': ETF[g][0], 'alternative': ETF[g][1],
                     'group_composition_aug2026': mix[g], 'caveat': ETF[g][2]} for g in P.GROUPS])
etf = pd.concat([etf, pd.DataFrame([{'group': 'hedge', 'name': 'Market hedge', 'etf': 'SPY', 'alternative': '',
                                     'group_composition_aug2026': '', 'caveat': 'Not in the frozen issuer-verified list; hedges Mkt-RF beta.'}])])
etf.to_csv(P.FORWARD / 'etf_proxies.csv', index=False)

book = pd.DataFrame({'group': list(P.GROUPS), 'name': [P.NAMES[g] for g in P.GROUPS], 'etf': [ETF[g][0] for g in P.GROUPS],
                     **{f'{v.lower()}_weight': TRANCHES[v].to_numpy() for v in ['V2a', 'V2b', 'V2c', 'V2d']}})
book.to_csv(P.FORWARD / 'book_2026Q4.csv', index=False)
weight_cols = [f'{v.lower()}_weight' for v in ['V2a', 'V2b', 'V2c', 'V2d']]
etf_book = book.groupby('etf')[weight_cols].sum()
etf_book.loc['SPY'] = [HEDGE[v]['spy_hedge_weight_per_unit'] for v in ['V2a', 'V2b', 'V2c', 'V2d']]
etf_book = etf_book[(etf_book.abs() > 1e-12).any(axis=1)]
etf_book.to_csv(P.FORWARD / 'book_2026Q4_etf.csv')
# The book was first built on 2026-10-03T09:17:55Z (commit 3e8ecd7). Later runs must reproduce it byte for byte;
# the recorded first-build time is then the build time reported in Gate 3.
first_build = P.FORWARD / 'book_first_build.sha256'
if first_build.exists():
    recorded_hash, recorded_utc = first_build.read_text().split()[0], first_build.read_text().splitlines()[1].strip()
    if sha256(P.FORWARD / 'book_2026Q4.csv') != recorded_hash:
        stop('analysis/output/forward/book_2026Q4.csv differs from the first build recorded in book_first_build.sha256')
    BOOK_UTC = recorded_utc
else:
    first_build.write_text(f"{sha256(P.FORWARD / 'book_2026Q4.csv')}  book_2026Q4.csv\n{BOOK_UTC}\n")
book

,group,name,etf,v2a_weight,v2b_weight,v2c_weight,v2d_weight
0,1,Mining & logging,XOP,-0.333333,0.000000,0.000000,-0.333333
1,2,Construction,ITB,0.333333,0.000000,0.333333,0.000000
2,3,Durable mfg,XLI,0.333333,0.333333,0.333333,-0.333333
3,4,Nondurable mfg,XLP,0.000000,0.000000,0.000000,0.000000
4,5,Wholesale,XLI,0.000000,0.000000,0.000000,0.000000
5,6,Retail,XRT,0.000000,0.333333,-0.333333,-0.333333
6,7,Transport & utilities,XLU,0.000000,0.333333,-0.333333,0.000000
7,8,Information,VOX,0.333333,0.000000,0.000000,0.000000
8,9,Finance & insurance,XLF,0.000000,0.000000,0.333333,0.333333
9,10,Real estate,IYR,0.000000,-0.333333,0.000000,0.333333


---
##### Gate 3
Every spec was frozen before its results existed, and every book is dollar neutral with three longs and three shorts.

In [28]:
ts = lambda s: datetime.fromisoformat(s)
checks3 = [
    ('v2 spec frozen before the v2 results', f'spec {SPEC_UTC} < first run {RESULTS_UTC}', ts(SPEC_UTC) < ts(RESULTS_UTC)),
    ('v2 spec unchanged since freezing', SPEC_HASH[:16], sha256(P.SPECS / 'study2_1_variants.md') == SPEC_HASH),
    ('Forward spec frozen before the book was built', f'spec {FWD_UTC} < book {BOOK_UTC}', ts(FWD_UTC) < ts(BOOK_UTC)),
    ('Forward spec unchanged since freezing', FWD_HASH[:16], sha256(P.SPECS / 'forward_spec.md') == FWD_HASH),
    ('Amendment 001 frozen after the v2 results and before the book', f'{RESULTS_UTC} < {AMD_UTC} < {BOOK_UTC}',
     ts(RESULTS_UTC) < ts(AMD_UTC) < ts(BOOK_UTC)),
    ('Amendment 001 unchanged since freezing', AMD_HASH[:16], sha256(P.SPECS / 'forward_amendment_001.md') == AMD_HASH),
    ('No observation after the fixed-lag month used (full vs truncated vintage)', 'identical scores, 4 variants',
     all(np.array_equal(scores_fwd[v].to_numpy(), scores_truncated[v].to_numpy()) for v in scores_fwd)),
]
for v, w in TRANCHES.items():
    checks3.append((f'{v} book weights net to zero', f'sum = {w.sum():+.1e}', abs(w.sum()) < 1e-12))
    checks3.append((f'{v} book has 3 longs and 3 shorts', f'{int((w > 0).sum())} long, {int((w < 0).sum())} short',
                    int((w > 0).sum()) == 3 and int((w < 0).sum()) == 3))
fingerprint_end, n_end = frozen_fingerprint()
checks3.append(('Frozen repo unchanged at the end of the run', f'{n_end} files, {fingerprint_end[:12]}...',
                (fingerprint_end, n_end) == P.FROZEN_FINGERPRINT))
gate3 = pd.DataFrame(checks3, columns=['check', 'value', 'pass'])
passed3 = bool(gate3['pass'].all())
(P.FORWARD / 'gate3.md').write_text(f'# Gate 3\n\nRun {utc_now()} by analysis/run.ipynb. **Result: {"PASS" if passed3 else "FAIL"}**\n\n'
                                 + G.markdown(gate3, index=False) + '\n')
if not passed3:
    stop('Gate 3 failed. See analysis/output/forward/gate3.md:\n\n' + G.markdown(gate3, index=False))
print('Gate 3:', 'PASS' if passed3 else 'FAIL')
gate3

Gate 3: PASS


,check,value,pass
0,v2 spec frozen before the v2 results,spec 2026-10-03T09:00:13+00:00 < first run 202...,True
1,v2 spec unchanged since freezing,ca6582347904489b,True
2,Forward spec frozen before the book was built,spec 2026-10-03T09:03:56+00:00 < book 2026-10-...,True
3,Forward spec unchanged since freezing,735a8ffa9c9c23fd,True
4,Amendment 001 frozen after the v2 results and ...,2026-10-03T09:01:55+00:00 < 2026-10-03T09:16:1...,True
5,Amendment 001 unchanged since freezing,c76cde86a85ff703,True
6,No observation after the fixed-lag month used ...,"identical scores, 4 variants",True
7,V2a book weights net to zero,sum = +0.0e+00,True
8,V2a book has 3 longs and 3 shorts,"3 long, 3 short",True
9,V2b book weights net to zero,sum = +1.1e-16,True


---
##### Study 2.2 freeze check
The robustness spec must be unchanged since it was hashed and committed before any check ran. POST-HOC.

In [29]:
v3_record = (P.SPECS / 'study2_2_hq12_robustness.sha256').read_text().split()
V3_HASH, V3_UTC = v3_record[0], v3_record[3]
if sha256(P.SPECS / 'study2_2_hq12_robustness.md') != V3_HASH:
    stop('analysis/specs/study2_2_hq12_robustness.md differs from the hash recorded in analysis/specs/study2_2_hq12_robustness.sha256')
N_TRIALS_V3 = 132                                   # 112 + 20 new portfolio variants (v3_spec.md section 1)
V2D_SCORE = V2['V2d'][0]
V2D_LEDGER = LEDGERS[('V2d', 10)]
H12 = D.horizon_target(R['total'], 12)
print('v3 spec frozen at', V3_UTC, 'sha256', V3_HASH[:16])


def ic12_stats(score, idx):
    return ic_stats(score, H12, idx, lags=17)


def perf_window(ledger, w_or_idx):
    idx = months(w_or_idx) if isinstance(w_or_idx, str) else w_or_idx
    return S.perf(ledger.reindex(idx).dropna(subset=['net']))

v3 spec frozen at 2026-10-03T09:52:27+00:00 sha256 312aef1dd9551aa7


---
##### R1 Components
Each labor flow alone, signed like HQ12 and held 12 months.

In [30]:
rows = []
for label, k in [('hires only', 'F2'), ('quits only', 'F3'), ('openings (contrast)', 'F1'), ('layoffs (contrast)', 'F4'), ('V2d = -(hires+quits)/2', None)]:
    score = V2D_SCORE if k is None else -Z[k]
    ledger = V2D_LEDGER if k is None else D.backtest(score.reindex(full_months), R, holding=12, cost_bps=10)[0]
    for w in ['research', 'test', 'full']:
        idx = months(w)
        pf = perf_window(ledger, w)
        m, se, t, n = ic12_stats(score, idx)
        fa = S.factor_attribution(ledger.reindex(idx).net, R)
        rows.append({'book': label, 'window': w, 'sharpe': pf['sharpe'], 'net_ann': pf['net_ann'], 'ic12': m, 'ic12_t': t, 'alpha_t': fa['t'], 'months': pf['n']})
r1 = pd.DataFrame(rows)
r1_show = r1.pivot(index='book', columns='window', values='sharpe')[['research', 'test', 'full']].reindex([r for r in dict.fromkeys(r1.book)])
r1_show = pd.DataFrame({f'Sharpe {w}': [f'{v:+.2f}' for v in r1_show[w]] for w in r1_show.columns}, index=r1_show.index)
r1_show['Test IC 12m (t)'] = [f"{a:+.3f} ({b:+.2f})" for a, b in r1[r1.window == 'test'].set_index('book').loc[r1_show.index, ['ic12', 'ic12_t']].to_numpy()]
r1_show['Test alpha t'] = [f'{v:+.2f}' for v in r1[r1.window == 'test'].set_index('book').loc[r1_show.index, 'alpha_t']]
r1_show.index.name = 'Book (held 12 months)'
G.write_table(r1, 'v3_r1_components', index=False, label=P.POSTHOC, display=r1_show, align='lrrrrr')
r1_show

,Sharpe research,Sharpe test,Sharpe full,Test IC 12m (t),Test alpha t
Book (held 12 months),,,,,
hires only,+0.41,+0.19,+0.27,+0.093 (+2.90),+0.79
quits only,+0.18,+0.16,+0.15,+0.085 (+1.75),+0.81
openings (contrast),+0.10,+0.42,+0.29,+0.082 (+1.38),+1.48
layoffs (contrast),+0.11,+0.16,+0.13,+0.032 (+0.87),+0.69
V2d = -(hires+quits)/2,+0.50,+0.29,+0.35,+0.096 (+2.43),+1.37


---
##### R2 Horizon profile
Rank IC of the HQ12 score against h-month relative returns, h = 1 to 24; the thesis figure.

In [31]:
rows = []
for h in [1, 3, 6, 9, 12, 18, 24]:
    target = D.horizon_target(R['total'], h)
    for w in ['research', 'test', 'full']:
        m, se, t, n = ic_stats(V2D_SCORE, target, months(w), lags=h + 5)
        rows.append({'window': w, 'h': h, 'ic': m, 'se': se, 't': t, 'months': n})
r2 = pd.DataFrame(rows)
r2_show = pd.DataFrame({w: [f"{a:+.3f} ({b:+.2f})" for a, b in r2[r2.window == w].sort_values('h')[['ic', 't']].to_numpy()] for w in ['research', 'test', 'full']},
                       index=pd.Index([f'{h}m' for h in sorted(r2.h.unique())], name='Horizon'))
G.write_table(r2, 'v3_r2_horizon_profile', index=False, label=P.POSTHOC, display=r2_show, align='lrrr')
G.horizon_profile(r2)
r2_show

,research,test,full
Horizon,,,
1m,-0.016 (-0.59),-0.020 (-0.87),-0.019 (-1.08)
3m,+0.005 (+0.18),+0.007 (+0.18),+0.005 (+0.21)
6m,-0.004 (-0.11),+0.017 (+0.40),+0.006 (+0.20)
9m,+0.023 (+0.48),+0.069 (+1.98),+0.046 (+1.52)
12m,+0.047 (+0.90),+0.096 (+2.43),+0.071 (+2.06)
18m,+0.038 (+0.59),+0.085 (+2.00),+0.059 (+1.41)
24m,+0.009 (+0.11),+0.063 (+1.86),+0.034 (+0.72)


---
##### R3 Holding period
The same score held 6, 9, 12 and 18 months.

In [32]:
rows = []
HOLD_LEDGERS = {}
for hold in [6, 9, 12, 18]:
    ledger = V2D_LEDGER if hold == 12 else D.backtest(V2D_SCORE.reindex(full_months), R, holding=hold, cost_bps=10)[0]
    HOLD_LEDGERS[hold] = ledger
    row = {'holding': hold, 'test_turnover': perf_window(ledger, 'test')['turnover']}
    for w in WINDOW_NAMES:
        row[w] = perf_window(ledger, w)['sharpe']
    rows.append(row)
r3 = pd.DataFrame(rows).set_index('holding')
r3_show = pd.DataFrame({w: [f'{v:+.2f}' for v in r3[w]] for w in WINDOW_NAMES}, index=pd.Index([f'{h} months' for h in r3.index], name='Holding'))
r3_show['Test turnover'] = [f'{v:.2f}' for v in r3.test_turnover]
G.write_table(r3, 'v3_r3_holding', label=P.POSTHOC, display=r3_show, align='lrrrrrr')
G.v2d_holding(r3[WINDOW_NAMES])
r3_show

,research,test,full,post-2010,last 18m,Test turnover
Holding,,,,,,
6 months,+0.31,+0.05,+0.15,+0.05,+0.50,0.49
9 months,+0.43,+0.14,+0.24,+0.16,+0.29,0.40
12 months,+0.50,+0.29,+0.35,+0.30,+0.29,0.37
18 months,+0.58,+0.30,+0.38,+0.34,+0.31,0.28


---
##### R4 Sub-periods
HQ12 Sharpe and 12-month IC in halves, excluding the pandemic, and before and after 2020.

In [33]:
test_idx, full_idx = months('test'), months('full')
pandemic = pd.period_range('2020-03', '2021-12', freq='M')
splits = {
    'test: first half': test_idx[:len(test_idx) // 2], 'test: second half': test_idx[len(test_idx) // 2:],
    'test: excl. Mar 2020-Dec 2021': test_idx.difference(pandemic), 'test: pre-2020': test_idx[test_idx < pd.Period('2020-01', freq='M')],
    'test: 2020 onward': test_idx[test_idx >= pd.Period('2020-01', freq='M')],
    'full: first half': full_idx[:len(full_idx) // 2], 'full: second half': full_idx[len(full_idx) // 2:],
    'full: excl. Mar 2020-Dec 2021': full_idx.difference(pandemic), 'full: pre-2020': full_idx[full_idx < pd.Period('2020-01', freq='M')],
}
rows = []
for name, idx in splits.items():
    pf = perf_window(V2D_LEDGER, idx)
    m, se, t, n = ic12_stats(V2D_SCORE, idx)
    rows.append({'split': name, 'months': pf['n'], 'sharpe': pf['sharpe'], 'net_ann': pf['net_ann'], 'ic12': m, 'ic12_t': t})
r4 = pd.DataFrame(rows).set_index('split')
r4_show = pd.DataFrame({'Months': r4.months.astype(int).to_numpy(), 'Sharpe': [f'{v:+.2f}' for v in r4.sharpe], 'Net p.a.': [f'{v:+.1%}' for v in r4.net_ann],
                        'IC 12m (t)': [f'{a:+.3f} ({b:+.2f})' for a, b in zip(r4.ic12, r4.ic12_t)]}, index=r4.index)
r4_show.index.name = 'Window'
G.write_table(r4, 'v3_r4_subperiods', label=P.POSTHOC, display=r4_show, align='lrrrr')
r4_show

,Months,Sharpe,Net p.a.,IC 12m (t)
Window,,,,
test: first half,71,-0.21,-1.4%,+0.097 (+1.75)
test: second half,71,+0.64,+6.5%,+0.095 (+1.60)
test: excl. Mar 2020-Dec 2021,120,+0.37,+3.0%,+0.105 (+2.23)
test: pre-2020,62,+0.14,+0.8%,+0.114 (+1.92)
test: 2020 onward,80,+0.37,+3.9%,+0.080 (+1.54)
full: first half,134,+0.33,+2.0%,+0.038 (+0.73)
full: second half,134,+0.38,+3.3%,+0.107 (+2.66)
full: excl. Mar 2020-Dec 2021,246,+0.40,+2.9%,+0.073 (+1.93)
full: pre-2020,188,+0.36,+2.2%,+0.068 (+1.57)


---
##### R5 Costs
Sharpe at 0, 10, 25 and 50 bp industry costs and the break-even cost.

In [34]:
cost_ledgers = {c: LEDGERS[('V2d', c)] for c in [0, 10, 25]}
cost_ledgers[50] = D.backtest(V2D_SCORE.reindex(full_months), R, holding=12, cost_bps=50)[0]
rows = []
for w in WINDOW_NAMES:
    idx = months(w)
    l0 = cost_ledgers[0].reindex(idx)
    row = {'window': w, **{f'sharpe_{c}bp': perf_window(l, w)['sharpe'] for c, l in cost_ledgers.items()},
           'breakeven_bp': 1e4 * l0.net.sum() / l0.turnover.sum()}
    rows.append(row)
r5 = pd.DataFrame(rows).set_index('window')
r5_show = pd.DataFrame({f'{c} bp': [f'{v:+.2f}' for v in r5[f'sharpe_{c}bp']] for c in [0, 10, 25, 50]}, index=r5.index)
r5_show['Break-even (bp)'] = [f'{v:.0f}' for v in r5.breakeven_bp]
r5_show.index.name = 'Window'
G.write_table(r5, 'v3_r5_costs', label=P.POSTHOC, display=r5_show, align='lrrrrr')
r5_show

,0 bp,10 bp,25 bp,50 bp,Break-even (bp)
Window,,,,,
research,+0.58,+0.50,+0.39,+0.20,76
test,+0.34,+0.29,+0.22,+0.09,68
full,+0.41,+0.35,+0.26,+0.11,69
post-2010,+0.35,+0.30,+0.21,+0.07,63
last 18m,+0.32,+0.29,+0.24,+0.16,99


---
##### R6 Factor alpha
FF5 + momentum + industry momentum with 12 HAC lags for the 12-month overlap; HQ12 added to the loadings figure.

In [35]:
rows = []
for w in ['test', 'full']:
    fa = S.factor_attribution(V2D_LEDGER.reindex(months(w)).net, R, lags=12)
    for f in ['intercept'] + FACTORS:
        rows.append({'window': w, 'factor': f, 'coef': fa['coef'][f], 'se': fa['se_all'][f], 't': fa['tstat'][f], 'n': fa['n']})
r6 = pd.DataFrame(rows)
r6_show = pd.DataFrame({w: [(f'{100 * c:+.2f}% ({t:+.2f})' if f == 'intercept' else f'{c:+.2f} ({t:+.2f})')
                            for f, c, t in r6.loc[r6.window == w, ['factor', 'coef', 't']].itertuples(index=False)] for w in ['test', 'full']},
                       index=pd.Index(['Alpha (per month)', 'Market', 'SMB', 'HML', 'RMW', 'CMA', 'Momentum', 'Industry momentum'], name='Factor'))
r6_show.columns = ['Test (HAC 12)', 'Full (HAC 12)']
G.write_table(r6, 'v3_r6_alpha', index=False, label=P.POSTHOC, display=r6_show, align='lrr')
v2d_load = r6[r6.window == 'test'].assign(strategy='V2d hires+quits 12m', label=P.POSTHOC)[['strategy', 'factor', 'coef', 'se', 't', 'n', 'label']]
G.loadings(pd.concat([d6, v2a_load, v2d_load])[lambda x: x.factor != 'intercept'])
r6_show

,Test (HAC 12),Full (HAC 12)
Factor,,
Alpha (per month),+0.26% (+1.48),+0.26% (+1.92)
Market,+0.03 (+0.58),-0.00 (-0.08)
SMB,-0.17 (-2.24),-0.02 (-0.31)
HML,+0.12 (+1.07),+0.10 (+1.23)
RMW,-0.36 (-4.45),-0.16 (-1.83)
CMA,-0.01 (-0.07),-0.09 (-0.87)
Momentum,-0.12 (-1.14),-0.03 (-0.51)
Industry momentum,+0.02 (+0.17),+0.06 (+0.93)


---
##### R7 Drop one industry
Thirteen reruns of HQ12, each with one group removed before standardisation and before the portfolio.

In [36]:
rows = []
for g in P.GROUPS:
    others = [x for x in P.GROUPS if x != g]
    Fg, _ = D.signals(groups=others)
    score_g = -(Fg['F2'] + Fg['F3']) / 2
    reduced = {'total': R['total'][others], 'excess': R['excess'][others], 'factors': R['factors']}
    reduced['relative'] = reduced['excess'].sub(reduced['excess'].mean(axis=1), axis=0)
    ledger_g = D.backtest(score_g.reindex(full_months), reduced, holding=12, cost_bps=10)[0]
    rows.append({'dropped_group': g, 'industry': P.NAMES[g], 'test_sharpe': perf_window(ledger_g, 'test')['sharpe'], 'full_sharpe': perf_window(ledger_g, 'full')['sharpe']})
r7 = pd.DataFrame(rows).set_index('industry')
baseline = {'test': perf_window(V2D_LEDGER, 'test')['sharpe'], 'full': perf_window(V2D_LEDGER, 'full')['sharpe']}
r7_summary = pd.DataFrame({'test_sharpe': [r7.test_sharpe.min(), r7.test_sharpe.median(), r7.test_sharpe.max(), baseline['test']],
                           'full_sharpe': [r7.full_sharpe.min(), r7.full_sharpe.median(), r7.full_sharpe.max(), baseline['full']]},
                          index=pd.Index(['minimum', 'median', 'maximum', 'all 13 groups (V2d)'], name='statistic'))
r7_summary.loc['industry whose removal lowers test Sharpe most'] = [r7.test_sharpe.idxmin(), '']
r7_summary.loc['industry whose removal raises test Sharpe most'] = [r7.test_sharpe.idxmax(), '']
r7_show = r7_summary.copy()
for c in ['test_sharpe', 'full_sharpe']:
    r7_show[c] = [f'{v:+.2f}' if isinstance(v, float) else v for v in r7_show[c]]
r7_show.columns = ['Test Sharpe', 'Full Sharpe']
G.write_table(r7, 'v3_r7_dropone', label=P.POSTHOC, fmt={'test_sharpe': '{:+.2f}', 'full_sharpe': '{:+.2f}'})
G.write_table(r7_summary, 'v3_r7_dropone_summary', label=P.POSTHOC, display=r7_show, align='lrr')
G.v2d_dropone(r7, baseline)
r7_show

,Test Sharpe,Full Sharpe
statistic,,
minimum,+0.18,+0.23
median,+0.30,+0.34
maximum,+0.48,+0.54
all 13 groups (V2d),+0.29,+0.35
industry whose removal lowers test Sharpe most,Health care,
industry whose removal raises test Sharpe most,Real estate,


---
##### R8 Tightness moderator
HQ12 conditional on the labor-market regime at the decision: T rising vs falling over 12 months, and V/U above vs below 1.

In [37]:
dT12 = (T - T.shift(12))
regimes = {'T rising': dT12 > 0, 'T falling': dT12 <= 0, 'V/U > 1': T > 0, 'V/U < 1': T <= 0}
rows = []
for w in ['test', 'full']:
    idx = months(w)
    ic12_series = S.rank_ic(V2D_SCORE.reindex(idx), H12.reindex(idx))
    for name, mask in regimes.items():
        sel = idx[mask.reindex(idx).fillna(False).to_numpy()]
        net = V2D_LEDGER.net.reindex(sel)
        h = S.hac(ic12_series.where(ic12_series.index.isin(sel)), lags=17)
        rows.append({'window': w, 'regime': name, 'n': len(sel), 'sharpe': float(np.sqrt(12) * net.mean() / net.std(ddof=1)) if len(sel) > 2 else np.nan,
                     'net_ann': float(12 * net.mean()), 'ic12': h['mean'], 'ic12_se': h['se'], 'ic12_t': h['t'], 'ic12_n': h['n']})
r8 = pd.DataFrame(rows)
r8_show = pd.DataFrame({'Months': r8.n.astype(int).to_numpy(), 'Sharpe': [f'{v:+.2f}' for v in r8.sharpe], 'Net p.a.': [f'{v:+.1%}' for v in r8.net_ann],
                        'IC 12m (t)': [f'{a:+.3f} ({b:+.2f})' for a, b in zip(r8.ic12, r8.ic12_t)]},
                       index=pd.Index([f'{w}: {r}' for w, r in zip(r8.window, r8.regime)], name='Window: regime'))
G.write_table(r8, 'v3_r8_tightness', index=False, label=P.POSTHOC, display=r8_show, align='lrrrr')
G.v2d_tightness(r8)
r8_show

,Months,Sharpe,Net p.a.,IC 12m (t)
Window: regime,,,,
test: T rising,88,-0.00,-0.0%,+0.115 (+2.52)
test: T falling,54,+0.66,+6.8%,+0.061 (+1.02)
test: V/U > 1,75,+0.72,+6.0%,+0.103 (+2.00)
test: V/U < 1,67,-0.14,-1.2%,+0.088 (+1.59)
full: T rising,182,+0.06,+0.4%,+0.048 (+1.19)
full: T falling,86,+0.83,+7.6%,+0.126 (+2.19)
full: V/U > 1,75,+0.72,+6.0%,+0.103 (+2.00)
full: V/U < 1,193,+0.19,+1.4%,+0.059 (+1.40)


---
##### R9 Bootstrap interval
Circular moving-block bootstrap (block 12, 5,000 draws, seed 230): 90% interval for the HQ12 Sharpe.

In [38]:
rows = []
for w in ['test', 'full']:
    bs = S.block_bootstrap_sharpe(V2D_LEDGER.reindex(months(w)).net, block=12, draws=5000, seed=P.SEED, level=0.90)
    rows.append({'window': w, **bs, 'excludes_zero': bool(bs['low'] > 0 or bs['high'] < 0)})
r9 = pd.DataFrame(rows).set_index('window')
r9_show = pd.DataFrame({'Sharpe': [f'{v:+.2f}' for v in r9.sharpe], '90% interval': [f'[{a:+.2f}, {b:+.2f}]' for a, b in zip(r9.low, r9.high)],
                        'Months': r9.n.astype(int).to_numpy(), 'Excludes 0': ['yes' if v else 'no' for v in r9.excludes_zero]}, index=r9.index)
r9_show.index.name = 'Window'
G.write_table(r9, 'v3_r9_bootstrap', label=P.POSTHOC, display=r9_show, align='lrrrr')
r9_show

,Sharpe,90% interval,Months,Excludes 0
Window,,,,
test,+0.29,"[-0.07, +0.65]",142,no
full,+0.35,"[+0.05, +0.64]",268,yes


---
##### R10 As-known data
Only with a FRED API key in analysis/.env; otherwise recorded as not run.

In [39]:
env = P.ROOT / '.env'
has_key = env.exists() and 'FRED_API_KEY' in env.read_text()
r10 = pd.DataFrame([{'check': 'R10 as-known (ALFRED vintage) rerun of V2a-V2d', 'status': 'run' if has_key else 'not run: no key',
                     'note': 'analysis/.env has no FRED_API_KEY; historical v2/v3 results use revised data with fixed lags (data_vintage.md)' if not has_key else ''}])
G.write_table(r10, 'v3_r10_asknown', index=False, label=P.POSTHOC, align='p{4.2cm}p{2.6cm}p{8.6cm}')
if has_key:
    raise NotImplementedError('ALFRED key rerun is not implemented in this notebook')
r10

,check,status,note
0,R10 as-known (ALFRED vintage) rerun of V2a-V2d,not run: no key,analysis/.env has no FRED_API_KEY; historical ...


---
##### R11 Fundamental law
Grinold-Kahn: implied IR = IC x sqrt(breadth) x TC for A3, W and HQ12 in the test window, against the realised Sharpe.

In [40]:
def transfer_coefficient(weights, score, idx):
    cors = []
    for m in idx:
        if m in weights.index and m in score.index:
            w, s = weights.loc[m, list(P.GROUPS)], score.loc[m]
            if np.isfinite(s).all() and w.abs().sum() > 0 and s.std() > 0:
                cors.append(np.corrcoef(w.to_numpy(), (s - s.mean()).to_numpy())[0, 1])
    return float(np.nanmean(cors)), len(cors)


a3_scores, a3_weights = D.saved_csv('ASOF_A3_scores.csv'), D.saved_csv('ASOF_A3_weights.csv')
rows = []
for name, score, weights, horizon_months, label in [('A3 primary', a3_scores, a3_weights, 1, P.CONFIRMATORY),
                                                     ('V2a wage growth', V2['V2a'][0], WEIGHTS['V2a'], 1, P.POSTHOC),
                                                     ('V2d -(hires+quits) 12m', V2D_SCORE, WEIGHTS['V2d'], 12, P.POSTHOC)]:
    idx = test
    target = R['relative'] if horizon_months == 1 else H12
    ic_mean = float(S.rank_ic(score.reindex(idx), target.reindex(idx)).mean())
    breadth = 13 * 12 / horizon_months
    tc, n_tc = transfer_coefficient(weights, score, idx)
    realised = S.perf((D.saved_csv('ASOF_A3_ledger.csv') if name.startswith('A3') else LEDGERS[(name[:3], 10)].reindex(idx)))['sharpe']
    rows.append({'book': name, 'label': label, 'ic': ic_mean, 'ic_horizon_months': horizon_months, 'breadth_per_year': breadth, 'tc': tc,
                 'implied_ir': ic_mean * np.sqrt(breadth) * tc, 'ceiling_ir_tc1': ic_mean * np.sqrt(breadth), 'realised_sharpe': realised})
r11 = pd.DataFrame(rows).set_index('book')
r11_show = pd.DataFrame({'IC': [f'{v:+.3f}' for v in r11.ic], 'Horizon': [f'{int(v)}m' for v in r11.ic_horizon_months], 'BR/yr': [f'{v:.0f}' for v in r11.breadth_per_year],
                         'TC': [f'{v:.2f}' for v in r11.tc], 'Implied IR': [f'{v:+.2f}' for v in r11.implied_ir], 'Ceiling (TC=1)': [f'{v:+.2f}' for v in r11.ceiling_ir_tc1],
                         'Realised Sharpe': [f'{v:+.2f}' for v in r11.realised_sharpe]}, index=r11.index)
r11_show.index.name = 'Book (test window)'
G.write_table(r11, 'v3_r11_fundamental_law', label=f'{P.CONFIRMATORY} (A3) / {P.POSTHOC}', display=r11_show, align='lrrrrrrr')
r11_show

,IC,Horizon,BR/yr,TC,Implied IR,Ceiling (TC=1),Realised Sharpe
Book (test window),,,,,,,
A3 primary,-0.021,1m,156,0.80,-0.21,-0.27,-0.58
V2a wage growth,+0.049,1m,156,0.91,+0.56,+0.62,+0.16
V2d -(hires+quits) 12m,+0.096,12m,13,0.48,+0.17,+0.35,+0.29


---
##### Study 2.2 deflation and reading
Deflated Sharpe with N_trials = 132 for every Study 2.1 variant, and the pre-declared reading rule for the battery.

In [41]:
rows = []
for v in V2:
    for w in ['test', 'full']:
        dsr = S.deflated_sharpe(LEDGERS[(v, 10)].reindex(months(w)).net, n_trials=N_TRIALS_V3)
        rows.append({'variant': v, 'window': w, 'sharpe': 12 * dsr['monthly_sharpe'] * np.sqrt(12) / 12, 'dsr_112': float(V2_STATS.set_index(['variant', 'window']).loc[(v, w), 'dsr']),
                     'dsr_132': dsr['dsr'], 'benchmark_annual_sharpe_132': dsr['benchmark_annual_sharpe']})
dsr132 = pd.DataFrame(rows)
dsr_show = pd.DataFrame({'Window': dsr132.window.to_numpy(), 'DSR (112 trials)': [f'{v:.2f}' for v in dsr132.dsr_112], 'DSR (132 trials)': [f'{v:.2f}' for v in dsr132.dsr_132],
                         'Benchmark Sharpe (132)': [f'{v:.2f}' for v in dsr132.benchmark_annual_sharpe_132]}, index=pd.Index(dsr132.variant, name='Variant'))
G.write_table(dsr132, 'v3_dsr132', index=False, label=P.POSTHOC, display=dsr_show, align='llrrr')

t_ = lambda frame, **kw: frame.loc[np.logical_and.reduce([frame[k] == v for k, v in kw.items()])].iloc[0]
rule = {
    'R1 hires-only and quits-only test Sharpe > 0': bool(t_(r1, book='hires only', window='test').sharpe > 0 and t_(r1, book='quits only', window='test').sharpe > 0),
    'R2 test 12m IC t >= 2 and 1m IC t < 2': bool(t_(r2, window='test', h=12).t >= 2 and t_(r2, window='test', h=1).t < 2),
    'R3 holding 9 and 18 months test Sharpe > 0': bool(r3.loc[9, 'test'] > 0 and r3.loc[18, 'test'] > 0),
    'R4 both test halves Sharpe > 0': bool(r4.loc['test: first half', 'sharpe'] > 0 and r4.loc['test: second half', 'sharpe'] > 0),
    'R5 test break-even cost > 25 bp': bool(r5.loc['test', 'breakeven_bp'] > 25),
    'R7 minimum drop-one test Sharpe > 0': bool(r7.test_sharpe.min() > 0),
    'R9 90% interval excludes 0 (test or full)': bool(r9.loc['test', 'excludes_zero'] or r9.loc['full', 'excludes_zero']),
}
reading = pd.DataFrame({'condition': list(rule), 'holds': list(rule.values())})
V3_READING = 'supported by the battery' if all(rule.values()) else 'fragile: ' + '; '.join(k.split(' ', 1)[0] for k, v in rule.items() if not v)
reading.to_csv(P.table_path('v3_reading'), index=False)
G.write_table(reading.assign(holds=reading.holds.map({True: 'yes', False: 'no'})), 'v3_reading', index=False, label=P.POSTHOC, align='lr')
V3_RESULTS_UTC = utc_now()
print('battery reading:', V3_READING)
reading

battery reading: fragile: R4


,condition,holds
0,R1 hires-only and quits-only test Sharpe > 0,True
1,R2 test 12m IC t >= 2 and 1m IC t < 2,True
2,R3 holding 9 and 18 months test Sharpe > 0,True
3,R4 both test halves Sharpe > 0,False
4,R5 test break-even cost > 25 bp,True
5,R7 minimum drop-one test Sharpe > 0,True
6,R9 90% interval excludes 0 (test or full),True


---
##### Study 2.2 results
Write analysis/output/study2/v3_results.md with every result and its reading.

In [42]:
v3_log = P.OUT2 / 'v3_run_log.md'
V3_FILES = ['v3_r1_components', 'v3_r2_horizon_profile', 'v3_r3_holding', 'v3_r4_subperiods', 'v3_r5_costs', 'v3_r6_alpha', 'v3_r7_dropone',
            'v3_r7_dropone_summary', 'v3_r8_tightness', 'v3_r9_bootstrap', 'v3_r10_asknown', 'v3_r11_fundamental_law', 'v3_dsr132', 'v3_reading']
if v3_log.exists():
    for name, digest in re.findall(r'\| `(tables/[^`]+)` \| `([0-9a-f]{64})` \|', v3_log.read_text()):
        if sha256(P.OUT2 / name) != digest:
            stop(f'v3 output {name} differs from the first run recorded in v3_run_log.md; v3 results must not change')
    V3_FIRST_UTC = re.search(r'FIRST_RUN_UTC (\S+)', v3_log.read_text()).group(1)
else:
    V3_FIRST_UTC = V3_RESULTS_UTC
    v3_log.write_text('# v3 run log\n\nThe robustness battery ran once, at **' + V3_FIRST_UTC + '** (spec frozen ' + V3_UTC + '). '
                      'Later executions must reproduce these files byte for byte.\n\n| file | sha256 of first run |\n|---|---|\n'
                      + '\n'.join(f'| `tables/{n}.csv` | `{sha256(P.table_path(n))}` |' for n in V3_FILES) + f'\n\nFIRST_RUN_UTC {V3_FIRST_UTC}\n')
lines = [f'# v3 results: robustness battery for V2d (POST-HOC)\n',
         f'First run **{V3_FIRST_UTC}** (regenerated {V3_RESULTS_UTC}, outputs verified identical to the first run via v3_run_log.md). '
         f'Spec `analysis/specs/study2_2_hq12_robustness.md` frozen at **{V3_UTC}** (sha256 `{V3_HASH}`). Deflated Sharpe ratios use N_trials = {N_TRIALS_V3}. '
         'Data: revised FRED values with fixed lags (`data_vintage.md`). Nothing here changes the confirmatory verdict (A3, Do not implement).\n',
         '## Reading against the pre-declared rule (v3_spec.md section 3)\n', f'**{V3_READING}**\n', G.markdown(reading, index=False), '']
for title, frame, name in [('R1 Components (held 12 months)', r1_show, 'v3_r1_components'), ('R2 Horizon profile: IC (t) of the V2d score', r2_show, 'v3_r2_horizon_profile'),
                           ('R3 Holding period: Sharpe by window', r3_show, 'v3_r3_holding'), ('R4 Sub-periods', r4_show, 'v3_r4_subperiods'),
                           ('R5 Costs', r5_show, 'v3_r5_costs'), ('R6 Factor regression, HAC 12 lags', r6_show, 'v3_r6_alpha'),
                           ('R7 Drop one industry (summary)', r7_show, 'v3_r7_dropone_summary'), ('R8 Tightness moderator', r8_show, 'v3_r8_tightness'),
                           ('R9 Block-bootstrap 90% interval', r9_show, 'v3_r9_bootstrap'), ('R10 As-known data', r10, 'v3_r10_asknown'),
                           ('R11 Fundamental law (test window)', r11_show, 'v3_r11_fundamental_law'), ('Deflated Sharpe at 112 vs 132 trials', dsr_show, 'v3_dsr132')]:
    lines += [f'## {title}\n', f'Source: `analysis/output/study2/tables/{name}.csv` (LaTeX `report/tables/{name}.tex`).\n',
              G.markdown(frame, index=name not in ('v3_r10_asknown',)), '']
lines += ['## R7 full list\n', G.markdown(r7.round(3)), '', '## Figures\n',
          '- `report/figures/fig_horizon_profile.pdf` (R2), `fig_v2d_holding.pdf` (R3), `fig_v2d_dropone.pdf` (R7), `fig_v2d_tightness.pdf` (R8); V2d added to `fig_loadings.pdf` (R6).', '']
(P.OUT2 / 'v3_results.md').write_text('\n'.join(lines))
print('v3_results.md written')

v3_results.md written


---
##### Gate 2
The Study 2.2 spec predates the first Study 2.2 output; the Study 2.1 result files are still byte-identical; the Study 1 folder is unchanged.

In [43]:
ts = lambda s: datetime.fromisoformat(s)
checks2 = [('v3 spec frozen before the first v3 output', f'{V3_UTC} < {V3_FIRST_UTC}', ts(V3_UTC) < ts(V3_FIRST_UTC)),
           ('v3 spec unchanged since freezing', V3_HASH[:16], sha256(P.SPECS / 'study2_2_hq12_robustness.md') == V3_HASH)]
log_text = (P.OUT2 / 'v2_run_log.md').read_text()
for name, digest in re.findall(r'\| `(tables/[^`]+)` \| `([0-9a-f]{64})` \|', log_text):
    checks2.append((f'v2 output byte-identical: {name}', digest[:16], sha256(P.OUT2 / name) == digest))
fingerprint_end, n_end = frozen_fingerprint()
checks2.append(('Frozen repo unchanged', f'{n_end} files, {fingerprint_end[:12]}...', (fingerprint_end, n_end) == P.FROZEN_FINGERPRINT))
gate2 = pd.DataFrame(checks2, columns=['check', 'value', 'pass'])
passed2 = bool(gate2['pass'].all())
(P.OUT2 / 'gate2.md').write_text(f'# Gate 2\n\nRun {utc_now()} by analysis/run.ipynb. **Result: {"PASS" if passed2 else "FAIL"}**\n\n' + G.markdown(gate2, index=False) + '\n')
if not passed2:
    stop('Gate 2 failed. See analysis/output/study2/gate2.md:\n\n' + G.markdown(gate2, index=False))
print('Gate 2:', 'PASS' if passed2 else 'FAIL')
gate2

Gate 2: PASS


,check,value,pass
0,v3 spec frozen before the first v3 output,2026-10-03T09:52:27+00:00 < 2026-10-03T09:53:3...,True
1,v3 spec unchanged since freezing,312aef1dd9551aa7,True
2,v2 output byte-identical: tables/v2_window_sta...,e4432b81a34866a9,True
3,v2 output byte-identical: tables/v2_reading_ru...,6f65790db7b6ae68,True
4,v2 output byte-identical: tables/v2b_signs.csv,3c2a387dd5c1f033,True
5,v2 output byte-identical: tables/v2_monthly_ne...,d3401a89700327a9,True
6,v2 output byte-identical: tables/v2_pnl_by_ind...,0dad6abdff45dd70,True
7,Frozen repo unchanged,"3067 files, eb109be96917...",True


---
##### Study 2.3 freeze check
The Study 2.3 specification must be unchanged since it was hashed and committed before any item ran; the variant backtest engine must reproduce Study 1's backtest. POST-HOC.

In [44]:
s23_record = (P.SPECS / 'study2_3_robustness.sha256').read_text().split()
S23_HASH, S23_UTC = s23_record[0], s23_record[3]
if sha256(P.SPECS / 'study2_3_robustness.md') != S23_HASH:
    stop('analysis/specs/study2_3_robustness.md differs from the hash recorded in study2_3_robustness.sha256')
N_TRIALS_23 = 162                                   # 132 + 30 new backtests (spec section 0)
OLD = {'W': 'V2a', 'SC': 'V2b', 'W+MOM': 'V2c', 'HQ12': 'V2d'}      # new name -> code inside the frozen result files
test_m = months('test')
A3_SCORES, A3_LEDGER, A3_WEIGHTS = D.saved_csv('ASOF_A3_scores.csv'), D.saved_csv('ASOF_A3_ledger.csv'), D.saved_csv('ASOF_A3_weights.csv')
A0_LEDGER = D.saved_csv('ASOF_A0_ledger.csv')
HQ12_SCORE, HQ12_LEDGER = V2D_SCORE, V2D_LEDGER
ENGINE = {('A3', 2.0, 'top3'): D.backtest_variant(A3_SCORES, R),
          ('HQ12', 2.0, 'top3'): D.backtest_variant(HQ12_SCORE.reindex(full_months), R, holding=12)}
engine_error = max(float((ENGINE[('A3', 2.0, 'top3')][0].net - A3_LEDGER.net).abs().max()),
                   float((ENGINE[('HQ12', 2.0, 'top3')][0].net - HQ12_LEDGER.net).abs().max()))
ENGINE_OK = engine_error < 1e-10
if not ENGINE_OK:
    stop(f'backtest_variant does not reproduce the Study 1 backtest (max difference {engine_error:.2e})')
print('Study 2.3 spec frozen at', S23_UTC, 'sha256', S23_HASH[:16], '| engine max difference', f'{engine_error:.1e}')

Study 2.3 spec frozen at 2026-10-04T09:18:39+00:00 sha256 068a3e0396332514 | engine max difference 9.9e-17


---
##### Study 1 robustness (A3 primary and A0 fixed rule)
One table from Study 1's saved outputs (CONFIRMATORY numbers, unchanged) plus the post-hoc diagnostics S1a to S1c of the Study 2.3 spec; the freeze record is checked against the files.

In [45]:
def fmt_s(v, d=2):
    return 'n/a' if v is None or (isinstance(v, float) and not np.isfinite(v)) else f'{v:+.{d}f}'


rows = []
def add(item, a3, a0, source, label=P.CONFIRMATORY):
    rows.append({'item': item, 'A3': a3, 'A0': a0, 'label': label, 'source': source})


var, win, att = res['variants'], res['windows'], res['attribution']
comp = {(c['comparison'], c['endpoint']): c for c in res['comparisons']}
for v, name in [('ASOF', 'as known'), ('FIRST', 'first release'), ('REVISED', 'revised'), ('FIXEDLAG', 'fixed lag')]:
    add(f'Test Sharpe, data {name}', fmt_s(var[v]['A3']['sharpe']), fmt_s(var[v]['A0']['sharpe']), f'results.json variants.{v}')
add('Net return p.a.', f"{var['ASOF']['A3']['mean_net_annual']:+.1%}", f"{var['ASOF']['A0']['mean_net_annual']:+.1%}", 'results.json variants.ASOF')
add('IC, one month (t)', f"{var['ASOF']['A3']['IC']['1']['mean']:+.3f} ({var['ASOF']['A3']['IC']['1']['t']:+.2f})",
    f"{var['ASOF']['A0']['IC']['1']['mean']:+.3f} ({var['ASOF']['A0']['IC']['1']['t']:+.2f})", 'results.json variants.ASOF.IC.1')
add('Factor alpha per month (t)', f"{100 * att['A3']['mean']:+.2f}% ({att['A3']['t']:+.2f})", f"{100 * att['A0']['mean']:+.2f}% ({att['A0']['t']:+.2f})", 'results.json attribution')
add('Placebo p (circular shift)', f"{res['placebo']['p']:.2f}", 'n/a (primary only)', 'results.json placebo')
add('Holm-adjusted p, IC and return', f"{comp[('A3 vs A1', 'IC')]['holm_p']:.2f}, {comp[('A3 vs A1', 'return')]['holm_p']:.2f} (A3 vs A1)",
    f"{comp[('A1 vs A0', 'IC')]['holm_p']:.2f}, {comp[('A1 vs A0', 'return')]['holm_p']:.2f} (A1 vs A0)", 'results.json comparisons')
add('Deflated Sharpe, research (41 trials)', f"{rs['strategies']['A3']['deflated_sharpe']['dsr']:.3f}", f"{rs['strategies']['A0']['deflated_sharpe']['dsr']:.3f}", 'research_summary.json')
add('Sharpe at 0 / 10 / 25 bp', f"{res['sensitivities']['cost_0']['sharpe']:+.2f} / {var['ASOF']['A3']['sharpe']:+.2f} / {res['sensitivities']['cost_25']['sharpe']:+.2f}",
    f"n/a / {var['ASOF']['A0']['sharpe']:+.2f} / n/a", 'results.json sensitivities (primary only)')
for key, name in [('first_half', 'first test half'), ('second_half', 'second test half'), ('ex_pandemic', 'excluding Mar-Dec 2020'), ('last_18', 'last 18 months')]:
    add(f'Sharpe, {name}', fmt_s(win['A3'][key]['sharpe']), fmt_s(win['A0'][key]['sharpe']), f'results.json windows.{key}')
drop = pd.Series({P.NAMES[int(g)]: v['sharpe'] for g, v in res['drop_one_group'].items()})
add('Drop one group, Sharpe range', f'{drop.min():+.2f} ({drop.idxmin()}) to {drop.max():+.2f} ({drop.idxmax()})', 'n/a (primary only)', 'results.json drop_one_group')
gates = res['gate']['criteria']
add('Gates G1 timing / G2 prediction / G3 alpha / G4 robustness / G5 implementable', ' / '.join('pass' if gates[g] else 'fail' for g in ['G1', 'G2', 'G3', 'G4', 'G5']), 'n/a (primary only)', 'results.json gate')
add('Verdict', res['gate']['verdict'], 'not primary', 'results.json gate')

# S1a block bootstrap of the saved test ledgers
BOOT1 = {s: S.block_bootstrap_sharpe(l.net, block=12, draws=5000, seed=P.SEED, level=0.90) for s, l in [('A3', A3_LEDGER), ('A0', A0_LEDGER)]}
add('Block-bootstrap 90% interval, test Sharpe', f"[{BOOT1['A3']['low']:+.2f}, {BOOT1['A3']['high']:+.2f}]", f"[{BOOT1['A0']['low']:+.2f}, {BOOT1['A0']['high']:+.2f}]",
    'saved ledgers; 12-month blocks, 5,000 draws, seed 230', P.POSTHOC)
# S1b horizon profile of the A3 score
hrows = []
for h in [1, 3, 6, 9, 12, 18, 24]:
    target_h = D.horizon_target(R['total'], h)
    ic_m, ic_se, ic_t, ic_n = ic_stats(A3_SCORES, target_h, test_m, lags=h + 5)
    hrows.append({'book': 'A3 agent features (Study 1 primary)', 'h': h, 'ic': ic_m, 'se': ic_se, 't': ic_t, 'months': ic_n})
    sub = r2[(r2.window == 'test') & (r2.h == h)].iloc[0]
    hrows.append({'book': 'HQ12 (Study 2, post-hoc)', 'h': h, 'ic': sub.ic, 'se': sub.se, 't': sub.t, 'months': sub.months})
d11_h = pd.DataFrame(hrows)
a3h = d11_h[d11_h.book.str.startswith('A3')].set_index('h')
add('A3 score IC at 1 / 6 / 12 / 24 months (t)', ' / '.join(f"{a3h.loc[h, 'ic']:+.3f} ({a3h.loc[h, 't']:+.2f})" for h in [1, 6, 12, 24]), 'n/a', 'd11_a3_horizon_profile.csv', P.POSTHOC)
G.write_table(d11_h, 'd11_a3_horizon_profile', index=False, label=P.POSTHOC,
              display=d11_h.assign(ic=[f'{a:+.3f} ({b:+.2f})' for a, b in zip(d11_h.ic, d11_h.t)], h=[f'{h}m' for h in d11_h.h]).pivot(index='h', columns='book', values='ic').reindex([f'{h}m' for h in [1, 3, 6, 9, 12, 18, 24]]).rename_axis('Horizon'), align='lrr')
a3_max_t = float(a3h.t.abs().max())
G.horizon_compare(d11_h, 'The A3 score predicts at no horizon; the HQ12 score from 9 months' if a3_max_t < 2 else 'Rank IC of the A3 and HQ12 scores by horizon')

# S1c cap variants and the rank-weighted A3 book (same engine as R2 and R3 below)
for cap in [3.0, np.inf]:
    ENGINE[('A3', cap, 'top3')] = D.backtest_variant(A3_SCORES, R, gross_cap=cap)
    ENGINE[('HQ12', cap, 'top3')] = D.backtest_variant(HQ12_SCORE.reindex(full_months), R, holding=12, gross_cap=cap)
ENGINE[('A3', 2.0, 'rank')] = D.backtest_variant(A3_SCORES, R, weighting='rank')
ENGINE[('HQ12', 2.0, 'rank')] = D.backtest_variant(HQ12_SCORE.reindex(full_months), R, holding=12, weighting='rank')
sharpe_t = lambda key: S.perf(ENGINE[key][0].reindex(test_m))['sharpe']
add('A3 Sharpe with cap 2 / cap 3 / no cap', ' / '.join(f"{sharpe_t(('A3', c, 'top3')):+.2f}" for c in [2.0, 3.0, np.inf]), 'n/a', 's23_r2_risk_overlay.csv', P.POSTHOC)
add('A3 Sharpe, rank-weighted book', f"{sharpe_t(('A3', 2.0, 'rank')):+.2f}", 'n/a', 's23_r3_rank_weighted.csv', P.POSTHOC)
d11 = pd.DataFrame(rows)
G.write_table(d11, 'd11_study1_robustness', index=False, label=f'{P.CONFIRMATORY} / {P.POSTHOC}',
              display=d11[['item', 'A3', 'A0']].rename(columns={'item': 'Check (test window, as-known data unless stated)', 'A3': 'A3 agent features (primary)', 'A0': 'A0 fixed rule'}), align='p{6.1cm}p{5.0cm}p{3.6cm}')

# Freeze record: hash of FREEZE.json, the opening timestamp, and the frozen file hashes against the repository.
fz = json.loads((P.SONNET / 'outputs/FREEZE.json').read_text())
ev = json.loads((P.SONNET / 'outputs/evaluation_started.json').read_text())
present = {k: v for k, v in fz['hashes'].items() if (P.SONNET / k).exists()}
matching = [k for k, v in present.items() if sha256(P.SONNET / k) == v]
different = sorted(set(present) - set(matching))
freeze = pd.DataFrame([
    ('SHA-256 of FREEZE.json equals evaluation_started.freeze_sha256', sha256(P.SONNET / 'outputs/FREEZE.json') == ev['freeze_sha256'], ev['freeze_sha256'][:16]),
    ('Freeze recorded (UTC)', True, fz['utc']), ('Test opened (UTC)', True, ev['utc']),
    ('Test opened after the freeze', datetime.fromisoformat(ev['utc']) > datetime.fromisoformat(fz['utc']),
     f"{(datetime.fromisoformat(ev['utc']) - datetime.fromisoformat(fz['utc'])).total_seconds():.3f} s later"),
    ('Files in the freeze inventory', True, len(fz['hashes'])), ('of which present in the repository', True, len(present)),
    ('present files whose hash matches the inventory', len(matching) == len(present) - len(different), len(matching)),
    ('present files that differ', True, '; '.join(different) if different else 'none'),
    ('First test decision in the freeze record', True, fz['first_sealed_decision'])], columns=['check', 'ok', 'value'])
G.write_table(freeze, 'd11_freeze_check', index=False, label=P.CONFIRMATORY, align='p{8cm}lp{6cm}')
d11[['item', 'A3', 'A0', 'label']]

,item,A3,A0,label
0,"Test Sharpe, data as known",-0.58,-0.36,CONFIRMATORY
1,"Test Sharpe, data first release",-0.52,-0.16,CONFIRMATORY
2,"Test Sharpe, data revised",-0.56,-0.26,CONFIRMATORY
3,"Test Sharpe, data fixed lag",-0.53,-0.27,CONFIRMATORY
4,Net return p.a.,-4.6%,-3.5%,CONFIRMATORY
5,"IC, one month (t)",-0.021 (-1.08),-0.020 (-0.71),CONFIRMATORY
6,Factor alpha per month (t),-0.38% (-1.86),-0.25% (-0.92),CONFIRMATORY
7,Placebo p (circular shift),0.72,n/a (primary only),CONFIRMATORY
8,"Holm-adjusted p, IC and return","1.00, 1.00 (A3 vs A1)","1.00, 1.00 (A1 vs A0)",CONFIRMATORY
9,"Deflated Sharpe, research (41 trials)",0.003,0.000,CONFIRMATORY


---
##### R1 As-known rerun of W, SC, W+MOM and HQ12
Signals as known at each decision date, from Alex's package; validated against his saved features and A0 scores before any variant is run. POST-HOC.

In [46]:
ASK_FILE, ASK_CHECK = P.OUT2 / 'asknown_features.csv', P.table_path('s23_r1_validation')
PKG_OUT = P.PACKAGE / 'studies/follow_the_workers/outputs'
if (PKG_OUT / 'panel_asof.parquet').exists():
    raw_ak = D.asknown_features()
    feat = pd.read_parquet(PKG_OUT / 'features_asof.parquet')
    feat['decision'] = pd.PeriodIndex(feat.decision, freq='M')
    both = raw_ak[raw_ak.feature != 'W'].merge(feat, on=['decision', 'group', 'feature'], how='outer', indicator=True)
    zf = {k: v.pivot(index='decision', columns='group', values='value') for k, v in feat.groupby('feature')}
    a0_ak = ((zf['F1'] + zf['F2'] - zf['F4'] + zf['F5']) / 4).set_axis(zf['F1'].index + 1)
    a0_saved = D.saved_csv('ASOF_A0_scores.csv')
    zf['W'] = D.standardize(raw_ak[raw_ak.feature == 'W'].pivot(index='decision', columns='group', values='raw'))
    z_long = pd.concat([v.stack(future_stack=True).rename('z').reset_index().assign(feature=k) for k, v in zf.items()])
    raw_ak.merge(z_long, on=['decision', 'group', 'feature'], how='left').sort_values(['feature', 'decision', 'group']).to_csv(ASK_FILE, index=False)
    pd.DataFrame([('raw F1-F5 rebuilt from panel_asof vs features_asof.raw_value: max abs difference', float((both.raw - both.raw_value).abs().max()), 1e-9),
                  ('rows without a counterpart or with a missing-value mismatch', float((both._merge != 'both').sum() + (both.raw.isna() != both.raw_value.isna()).sum()), 0.5),
                  ('A0 from features_asof vs saved ASOF_A0_scores.csv, test window: max abs difference', float((a0_ak.reindex(a0_saved.index) - a0_saved).abs().max().max()), 1e-9)],
                 columns=['check', 'value', 'tolerance']).to_csv(ASK_CHECK, index=False)
    print('as-known features rebuilt from the package')
else:
    print('package not present: using the saved as-known features (analysis/output/study2/asknown_features.csv)')
ak_check = pd.read_csv(ASK_CHECK)
R1_OK = bool((ak_check.value < ak_check.tolerance).all())
ak = pd.read_csv(ASK_FILE)
ak['decision'] = pd.PeriodIndex(ak.decision, freq='M')
ZA = {k: v.pivot(index='decision', columns='group', values='z').pipe(lambda x: x.set_axis(x.index + 1)) for k, v in ak.groupby('feature')}        # earning month
RAWA = {k: v.pivot(index='decision', columns='group', values='raw') for k, v in ak.groupby('feature')}                                          # decision month
print('R1 validation passed:', R1_OK)
ak_check

as-known features rebuilt from the package
R1 validation passed: True


,check,value,tolerance
0,raw F1-F5 rebuilt from panel_asof vs features_...,0.000000e+00,1.000000e-09
1,rows without a counterpart or with a missing-v...,0.000000e+00,5.000000e-01
2,A0 from features_asof vs saved ASOF_A0_scores....,4.440892e-16,1.000000e-09


In [47]:
if not R1_OK:
    stop('R1 validation failed; as-known rerun not run (spec section 2)')
res_m = months('research')
AK_RES_IC = {k: float(S.rank_ic(ZA[k].reindex(res_m), R['relative'].reindex(res_m)).mean()) for k in ['F1', 'F2', 'F3', 'F4', 'F5', 'W']}
AK_SIGNS = {k: (1.0 if v >= 0 else -1.0) for k, v in AK_RES_IC.items()}
V23 = {'W': (D.csrank(ZA['W']), 3), 'SC': (sum(AK_SIGNS[k] * ZA[k] for k in AK_SIGNS) / 6, 3),
       'W+MOM': ((D.csrank(ZA['W']) + D.csrank(F['IndMom'])) / 2, 3), 'HQ12': (-(ZA['F2'] + ZA['F3']) / 2, 12)}
AK_LEDGERS, AK_WEIGHTS = {}, {}
for v, (score, holding) in V23.items():
    AK_LEDGERS[(v, 10)], AK_WEIGHTS[v] = D.backtest(score.reindex(full_months), R, holding=holding, cost_bps=10)
for cost in [0, 25, 50]:
    AK_LEDGERS[('HQ12', cost)] = D.backtest(V23['HQ12'][0].reindex(full_months), R, holding=12, cost_bps=cost)[0]

top3 = lambda frame: frame.apply(lambda s: frozenset(s.sort_index().sort_values(kind='stable').index[-3:]), axis=1)
bot3 = lambda frame: frame.apply(lambda s: frozenset(s.sort_index().sort_values(kind='stable').index[:3]), axis=1)
rows, overlap = [], []
fl = V2_STATS.set_index(['variant', 'window'])
for v, (score, holding) in V23.items():
    h = 1 if holding == 3 else 12
    target_h = R['relative'] if h == 1 else H12
    for w in WINDOW_NAMES:
        idx = months(w)
        l = AK_LEDGERS[(v, 10)].reindex(idx)
        ic_m, ic_se, ic_t, ic_n = ic_stats(score, target_h, idx, lags=h + 5)
        fa = S.factor_attribution(l.net, R, lags=6 if h == 1 else 12)
        rows.append({'variant': v, 'window': w, 'sharpe_asknown': S.perf(l)['sharpe'], 'sharpe_fixedlag': float(fl.loc[(OLD[v], w), 'sharpe']),
                     'ic_horizon': h, 'ic_asknown': ic_m, 'ic_t_asknown': ic_t,
                     'ic_fixedlag': float(fl.loc[(OLD[v], w), 'ic' if h == 1 else 'ic_hold']), 'ic_t_fixedlag': float(fl.loc[(OLD[v], w), 'ic_t' if h == 1 else 'ic_hold_t']),
                     'alpha_t_asknown': fa['t'], 'dsr_162_asknown': S.deflated_sharpe(l.net, n_trials=N_TRIALS_23)['dsr']})
    a, b = score.reindex(test_m).dropna(), V2[OLD[v]][0].reindex(test_m).dropna()
    common = a.index.intersection(b.index)
    overlap.append({'variant': v, 'test_months': len(common), 'same_top3': float((top3(a.loc[common]) == top3(b.loc[common])).mean()),
                    'same_bottom3': float((bot3(a.loc[common]) == bot3(b.loc[common])).mean()),
                    'score_rank_corr': float(S.rank_ic(a.loc[common], b.loc[common]).mean())})
s23_r1 = pd.DataFrame(rows)
s23_ov = pd.DataFrame(overlap).set_index('variant')
t1 = s23_r1[s23_r1.window == 'test'].set_index('variant')
r1_show = pd.DataFrame({'Sharpe, fixed lag': [f'{v:+.2f}' for v in t1.sharpe_fixedlag], 'Sharpe, as known': [f'{v:+.2f}' for v in t1.sharpe_asknown],
                        'IC (t), fixed lag': [f'{a:+.3f} ({b:+.2f})' for a, b in zip(t1.ic_fixedlag, t1.ic_t_fixedlag)],
                        'IC (t), as known': [f'{a:+.3f} ({b:+.2f})' for a, b in zip(t1.ic_asknown, t1.ic_t_asknown)],
                        'Alpha t': [f'{v:+.2f}' for v in t1.alpha_t_asknown], 'Same top 3': [f'{v:.0%}' for v in s23_ov.same_top3], 'Same bottom 3': [f'{v:.0%}' for v in s23_ov.same_bottom3]},
                       index=pd.Index(t1.index, name='Variant (test window)'))
G.write_table(s23_r1, 's23_r1_variants', index=False, label=P.POSTHOC, display=r1_show.reset_index(), align='lrrrrrrr')
s23_ov.to_csv(P.table_path('s23_r1_overlap'))
pd.DataFrame({'research_ic': AK_RES_IC, 'sign': AK_SIGNS}).rename_axis('signal').to_csv(P.table_path('s23_r1_sc_signs'))
r1_show

,"Sharpe, fixed lag","Sharpe, as known","IC (t), fixed lag","IC (t), as known",Alpha t,Same top 3,Same bottom 3
Variant (test window),,,,,,,
W,+0.16,+0.13,+0.049 (+2.04),+0.041 (+1.73),+0.18,76%,70%
SC,-0.15,+0.24,+0.042 (+1.65),+0.021 (+0.75),+0.71,0%,0%
W+MOM,+0.35,+0.46,+0.032 (+1.21),+0.036 (+1.40),+1.93,84%,74%
HQ12,+0.29,+0.20,+0.096 (+2.43),+0.085 (+2.26),+0.92,39%,51%


In [48]:
# HQ12 battery as known, next to the fixed-lag numbers of Study 2.2.
AKS, AKL = V23['HQ12'][0], AK_LEDGERS[('HQ12', 10)]
bat = []
def bat_add(check, fixed, known):
    bat.append({'check': check, 'fixed_lag': fixed, 'as_known': known})


sh = lambda ledger, idx: S.perf(ledger.reindex(idx).dropna(subset=['net']))['sharpe']
bat_add('Test Sharpe (10 bp)', sh(HQ12_LEDGER, test_m), sh(AKL, test_m))
bat_add('Full-sample Sharpe (10 bp)', sh(HQ12_LEDGER, full_months), sh(AKL, full_months))
bat_add('12-month IC, test', ic12_stats(HQ12_SCORE, test_m)[0], ic12_stats(AKS, test_m)[0])
bat_add('12-month IC t, test', ic12_stats(HQ12_SCORE, test_m)[2], ic12_stats(AKS, test_m)[2])
bat_add('1-month IC t, test', ic_stats(HQ12_SCORE, R['relative'], test_m)[2], ic_stats(AKS, R['relative'], test_m)[2])
for label, k in [('hires only', 'F2'), ('quits only', 'F3')]:
    l_k = D.backtest((-ZA[k]).reindex(full_months), R, holding=12, cost_bps=10)[0]
    bat_add(f'Component {label}, test Sharpe', float(t_(r1, book=label, window='test').sharpe), sh(l_k, test_m))
for hold in [9, 18]:
    l_h = D.backtest(AKS.reindex(full_months), R, holding=hold, cost_bps=10)[0]
    bat_add(f'Holding {hold} months, test Sharpe', float(r3.loc[hold, 'test']), sh(l_h, test_m))
half = len(test_m) // 2
bat_add('First test half, Sharpe', float(r4.loc['test: first half', 'sharpe']), sh(AKL, test_m[:half]))
bat_add('Second test half, Sharpe', float(r4.loc['test: second half', 'sharpe']), sh(AKL, test_m[half:]))
for cost in [0, 25, 50]:
    bat_add(f'Test Sharpe at {cost} bp', float(r5.loc['test', f'sharpe_{cost}bp']), sh(AK_LEDGERS[('HQ12', cost)], test_m))
l0 = AK_LEDGERS[('HQ12', 0)].reindex(test_m)
bat_add('Break-even cost, test (bp)', float(r5.loc['test', 'breakeven_bp']), 1e4 * l0.net.sum() / l0.turnover.sum())
drop_ak = {}
for g in P.GROUPS:
    others = [x for x in P.GROUPS if x != g]
    zg = {k: D.standardize(RAWA[k][others]).pipe(lambda x: x.set_axis(x.index + 1)) for k in ['F2', 'F3']}
    reduced = {'total': R['total'][others], 'excess': R['excess'][others], 'factors': R['factors']}
    drop_ak[P.NAMES[g]] = sh(D.backtest((-(zg['F2'] + zg['F3']) / 2).reindex(full_months), reduced, holding=12, cost_bps=10)[0], test_m)
drop_ak = pd.Series(drop_ak)
bat_add('Drop one group, minimum test Sharpe', float(r7.test_sharpe.min()), float(drop_ak.min()))
bat_add('Drop one group, maximum test Sharpe', float(r7.test_sharpe.max()), float(drop_ak.max()))
for w in ['test', 'full']:
    bs = S.block_bootstrap_sharpe(AKL.reindex(months(w)).net, block=12, draws=5000, seed=P.SEED, level=0.90)
    bat_add(f'Bootstrap 90% interval, {w}: low', float(r9.loc[w, 'low']), bs['low'])
    bat_add(f'Bootstrap 90% interval, {w}: high', float(r9.loc[w, 'high']), bs['high'])
    fa = S.factor_attribution(AKL.reindex(months(w)).net, R, lags=12)
    bat_add(f'Alpha per month (%), {w}, HAC 12', 100 * float(t_(r6, window=w, factor='intercept').coef), 100 * fa['mean'])
    bat_add(f'Alpha t, {w}, HAC 12', float(t_(r6, window=w, factor='intercept').t), fa['t'])
bat_add('Deflated Sharpe, test (132 looks fixed lag; 162 as known)', float(t_(dsr132, variant='V2d', window='test').dsr_132), S.deflated_sharpe(AKL.reindex(test_m).net, n_trials=N_TRIALS_23)['dsr'])
s23_bat = pd.DataFrame(bat).set_index('check')
hq = t1.loc['HQ12']
same_sign = np.sign(hq.sharpe_asknown) == np.sign(hq.sharpe_fixedlag) and np.sign(hq.ic_asknown) == np.sign(hq.ic_fixedlag)
R1_READING = 'as-known confirms the fixed-lag reading' if same_sign and abs(hq.sharpe_asknown - hq.sharpe_fixedlag) <= 0.15 else 'as-known differs'
G.write_table(s23_bat, 's23_r1_hq12_battery', label=P.POSTHOC, fmt={'fixed_lag': '{:+.2f}', 'as_known': '{:+.2f}'}, align='lrr',
              display=s23_bat.apply(lambda col: [f'{v:+.2f}' if abs(v) < 5 else f'{v:.0f}' for v in col]).rename(columns={'fixed_lag': 'Fixed lag (Study 2.2)', 'as_known': 'As known (Study 2.3)'}).rename_axis('HQ12 check'))
drop_ak.rename('test_sharpe').rename_axis('dropped').to_csv(P.table_path('s23_r1_hq12_dropone'))
print('R1 reading:', R1_READING)
s23_bat.round(3)

R1 reading: as-known confirms the fixed-lag reading


,fixed_lag,as_known
check,,
Test Sharpe (10 bp),0.293,0.204
Full-sample Sharpe (10 bp),0.351,0.312
"12-month IC, test",0.096,0.085
"12-month IC t, test",2.425,2.256
"1-month IC t, test",-0.868,-0.422
"Component hires only, test Sharpe",0.189,0.189
"Component quits only, test Sharpe",0.160,0.168
"Holding 9 months, test Sharpe",0.140,0.155
"Holding 18 months, test Sharpe",0.302,0.108


---
##### R2 Risk overlay and R3 rank-weighted books
Why the cap binds: the ex-ante volatility of the unit book and the gross exposure needed to reach 10%; A3 and HQ12 with the cap at 2, 3 and none; then rank-weighted tranches. POST-HOC.

In [49]:
rows, dist, needed = [], [], []
for book in ['A3', 'HQ12']:
    base = ENGINE[(book, 2.0, 'top3')][0].reindex(test_m)
    needed.append(pd.DataFrame({'book': 'A3 agent features (Study 1)' if book == 'A3' else 'HQ12 (Study 2)', 'gross_needed': base.gross_needed.to_numpy()}))
    dist.append({'book': book, **{f'unit_vol_p{q}': float(base.unit_vol.quantile(q / 100)) for q in [5, 50, 95]},
                 **{f'gross_needed_p{q}': float(base.gross_needed.quantile(q / 100)) for q in [5, 50, 95]}})
    for cap in [2.0, 3.0, np.inf]:
        l = ENGINE[(book, cap, 'top3')][0].reindex(test_m)
        pf = S.perf(l)
        rows.append({'book': book, 'cap': 'none' if np.isinf(cap) else f'{cap:.0f}', 'sharpe': pf['sharpe'], 'net_ann': pf['net_ann'], 'realised_vol': pf['vol'],
                     'mean_gross': float(l.gross_leverage.mean()), 'cap_binds_share': float(l.cap_binds.mean()), 'max_dd': pf['max_dd']})
s23_r2, s23_r2d = pd.DataFrame(rows), pd.DataFrame(dist).set_index('book')
r2o_show = pd.DataFrame({'Book': s23_r2.book.to_numpy(), 'Cap': s23_r2.cap.to_numpy(), 'Sharpe': [f'{v:+.2f}' for v in s23_r2.sharpe], 'Realised vol': [f'{v:.1%}' for v in s23_r2.realised_vol],
                         'Mean gross': [f'{v:.2f}' for v in s23_r2.mean_gross], 'Cap binds': [f'{v:.0%}' for v in s23_r2.cap_binds_share], 'Max drawdown': [f'{v:.1%}' for v in s23_r2.max_dd]})
G.write_table(s23_r2, 's23_r2_risk_overlay', index=False, label=f'{P.POSTHOC} (diagnostic of a confirmatory book for A3)', display=r2o_show, align='llrrrrr')
G.write_table(s23_r2d, 's23_r2_distribution', label=P.POSTHOC, fmt={c: '{:.3f}' if c.startswith('unit') else '{:.2f}' for c in s23_r2d.columns})
G.gross_needed(pd.concat(needed), 'Reaching 10% volatility would need more than twice gross in most months')

rows = []
a3_ic = float(S.rank_ic(A3_SCORES.reindex(test_m), R['relative'].reindex(test_m)).mean())
hq_ic = float(S.rank_ic(HQ12_SCORE.reindex(test_m), H12.reindex(test_m)).mean())
for book, score, ic_mean, breadth in [('A3', A3_SCORES, a3_ic, 156.0), ('HQ12', HQ12_SCORE, hq_ic, 13.0)]:
    for weighting in ['top3', 'rank']:
        l, w = ENGINE[(book, 2.0, weighting)]
        pf = S.perf(l.reindex(test_m))
        tc, _ = transfer_coefficient(w, score, test_m)
        rows.append({'book': book, 'weighting': 'top 3 / bottom 3' if weighting == 'top3' else 'rank-weighted', 'sharpe': pf['sharpe'], 'net_ann': pf['net_ann'], 'tc': tc,
                     'turnover': pf['turnover'], 'ic': ic_mean, 'breadth_per_year': breadth, 'implied_ir': ic_mean * np.sqrt(breadth) * tc,
                     'dsr_162': S.deflated_sharpe(l.reindex(test_m).net, n_trials=N_TRIALS_23)['dsr']})
s23_r3 = pd.DataFrame(rows)
r3o_show = pd.DataFrame({'Book': s23_r3.book.to_numpy(), 'Weights': s23_r3.weighting.to_numpy(), 'Test Sharpe': [f'{v:+.2f}' for v in s23_r3.sharpe], 'TC': [f'{v:.2f}' for v in s23_r3.tc],
                         'Turnover': [f'{v:.2f}' for v in s23_r3.turnover], 'Implied IR': [f'{v:+.2f}' for v in s23_r3.implied_ir]})
G.write_table(s23_r3, 's23_r3_rank_weighted', index=False, label=P.POSTHOC, display=r3o_show, align='llrrrr')
r2o_show

,Book,Cap,Sharpe,Realised vol,Mean gross,Cap binds,Max drawdown
0,A3,2,-0.58,7.8%,1.98,87%,-41.1%
1,A3,3,-0.61,9.7%,2.57,35%,-50.7%
2,A3,none,-0.62,10.3%,2.79,0%,-54.9%
3,HQ12,2,+0.29,8.7%,1.97,83%,-14.4%
4,HQ12,3,+0.24,10.5%,2.51,39%,-20.3%
5,HQ12,none,+0.24,11.1%,2.78,0%,-22.6%


---
##### R4 Agents: arm gap against seed spread, and R5 tokens
Both agent studies, per arm and seed, with an exact permutation test (20 relabelings); tokens per call and their rank correlation with research IC. POST-HOC, research data only.

In [50]:
import ast
import itertools
rows, token_rows, perm_rows, rho_rows = [], [], [], []
for study, root in [('Study 1 (Sonnet)', P.SONNET), ('Study 1b (Opus)', P.OPUS)]:
    cand = pd.read_csv(root / 'outputs/candidates.csv')
    parsed = [json.loads(s) if isinstance(s, str) else None for s in cand.spec]
    cand['expression'] = [p.get('expression', '') if isinstance(p, dict) and isinstance(p.get('expression'), str) else '' for p in parsed]
    eligible = {d['candidate_id']: d['eligible'] for arm_d in json.loads((root / 'outputs/selected_features.json').read_text())['decisions'].values() for d in arm_d}
    for (arm, seed), g in cand.groupby(['arm', 'seed']):
        valid = g[g.valid & g.research_mean_ic.notna()]
        rows.append({'study': study, 'arm': arm, 'seed': seed, 'proposals': len(g), 'valid': int(g.valid.sum()), 'best_ic': float(valid.research_mean_ic.max()),
                     'mean_ic': float(valid.research_mean_ic.mean()), 'eligible': (sum(bool(eligible.get(c)) for c in g.candidate_id) if seed == 1 else np.nan),
                     'regime_share': float(valid.expression.str.contains(r'regime\(').mean())})
    log = [json.loads(line) for line in (root / 'outputs/agents/log.jsonl').read_text().splitlines()]
    usage = {}
    for e in log:
        u = e.get('tokens')
        u = ast.literal_eval(u) if isinstance(u, str) and u.startswith('{') else u
        if e.get('kind') == 'candidate' and isinstance(u, dict):
            cid = e['call_id'].replace('_syntax_retry', '')
            total = sum(v for k, v in u.items() if k in ('input_tokens', 'cache_creation_input_tokens', 'cache_read_input_tokens', 'output_tokens') and isinstance(v, (int, float)))
            prev = usage.get(cid, {'total': 0, 'output': 0})
            usage[cid] = {'total': prev['total'] + total, 'output': prev['output'] + u.get('output_tokens', 0)}
    cand['tokens_total'] = cand.candidate_id.map(lambda c: usage.get(c, {}).get('total', np.nan))
    cand['tokens_output'] = cand.candidate_id.map(lambda c: usage.get(c, {}).get('output', np.nan))
    for arm, g in cand.groupby('arm'):
        token_rows.append({'study': study, 'arm': arm, 'calls_with_usage': int(g.tokens_total.notna().sum()), 'tokens_total': float(g.tokens_total.sum()),
                           'tokens_per_call': float(g.tokens_total.mean()), 'output_tokens_per_call': float(g.tokens_output.mean())})
    v = cand[cand.valid & cand.research_mean_ic.notna() & cand.tokens_output.notna()]
    rho_rows.append({'study': study, 'n': len(v), 'spearman_output_tokens_vs_ic': float(v.tokens_output.corr(v.research_mean_ic, method='spearman')),
                     'spearman_output_tokens_vs_abs_ic': float(v.tokens_output.corr(v.research_mean_ic.abs(), method='spearman'))})
s23_r4 = pd.DataFrame(rows)
for study, g in s23_r4.groupby('study', sort=False):
    for metric in ['best_ic', 'mean_ic']:
        values, arms = g[metric].to_numpy(), (g.arm == 'islands').to_numpy()
        observed = values[arms].mean() - values[~arms].mean()
        diffs = [values[list(c)].mean() - values[[i for i in range(6) if i not in c]].mean() for c in itertools.combinations(range(6), 3)]
        seed_range = max(values[arms].max() - values[arms].min(), values[~arms].max() - values[~arms].min())
        perm_rows.append({'study': study, 'metric': metric, 'islands_minus_independent': observed, 'largest_seed_range_within_arm': seed_range,
                          'gap_within_seed_range': bool(abs(observed) <= seed_range), 'perm_p_two_sided': float(np.mean(np.abs(diffs) >= abs(observed) - 1e-12)), 'relabelings': len(diffs)})
s23_r4p, s23_r5, s23_r5r = pd.DataFrame(perm_rows), pd.DataFrame(token_rows), pd.DataFrame(rho_rows)
r4_show4 = pd.DataFrame({'Study': s23_r4.study.to_numpy(), 'Arm': s23_r4.arm.to_numpy(), 'Seed': s23_r4.seed.to_numpy(), 'Valid': [f'{v}/{n}' for v, n in zip(s23_r4.valid, s23_r4.proposals)],
                         'Best IC': [f'{v:+.3f}' for v in s23_r4.best_ic], 'Mean IC': [f'{v:+.3f}' for v in s23_r4.mean_ic],
                         'Eligible': ['' if np.isnan(v) else f'{int(v)}' for v in s23_r4.eligible], 'Regime-gated': [f'{v:.0%}' for v in s23_r4.regime_share]})
G.write_table(s23_r4, 's23_r4_agents_by_seed', index=False, label=P.POSTHOC, display=r4_show4, align='llrrrrrr')
G.write_table(s23_r4p, 's23_r4_permutation', index=False, label=P.POSTHOC, align='llrrlr', display=pd.DataFrame({'Study': s23_r4p.study.to_numpy(), 'Metric': s23_r4p.metric.map({'best_ic': 'best IC', 'mean_ic': 'mean IC'}).to_numpy(),
              'Communicating minus independent': [f'{v:+.3f}' for v in s23_r4p.islands_minus_independent], 'Largest seed range': [f'{v:.3f}' for v in s23_r4p.largest_seed_range_within_arm],
              'Gap within seed range': ['yes' if v else 'no' for v in s23_r4p.gap_within_seed_range], 'Permutation p': [f'{v:.2f}' for v in s23_r4p.perm_p_two_sided]}))
R5_OK = bool(s23_r5.calls_with_usage.sum() > 0)
G.write_table(s23_r5, 's23_r5_tokens', index=False, label=P.POSTHOC, align='llrrrr', display=pd.DataFrame({'Study': s23_r5.study.to_numpy(), 'Arm': s23_r5.arm.to_numpy(), 'Calls': s23_r5.calls_with_usage.to_numpy(),
              'Tokens, total': [f'{v:,.0f}' for v in s23_r5.tokens_total], 'Tokens per call': [f'{v:,.0f}' for v in s23_r5.tokens_per_call], 'Output tokens per call': [f'{v:,.0f}' for v in s23_r5.output_tokens_per_call]}))
G.write_table(s23_r5r, 's23_r5_tokens_ic', index=False, label=P.POSTHOC, align='lrrr', display=pd.DataFrame({'Study': s23_r5r.study.to_numpy(), 'Proposals': s23_r5r.n.to_numpy(),
              'Spearman, output tokens vs IC': [f'{v:+.2f}' for v in s23_r5r.spearman_output_tokens_vs_ic], 'Spearman, output tokens vs absolute IC': [f'{v:+.2f}' for v in s23_r5r.spearman_output_tokens_vs_abs_ic]}))
within = bool(s23_r4p.gap_within_seed_range.all())
G.agents_seeds(s23_r4, 'The gap between arms is within the spread across seeds' if within else 'Best research IC per seed, by study and arm')
print('arm gap within seed range for every study and metric:', within, '| tokens logged:', R5_OK)
s23_r4p

arm gap within seed range for every study and metric: True | tokens logged: True


,study,metric,islands_minus_independent,largest_seed_range_within_arm,gap_within_seed_range,perm_p_two_sided,relabelings
0,Study 1 (Sonnet),best_ic,-0.034222,0.055985,True,0.3,20
1,Study 1 (Sonnet),mean_ic,-0.000562,0.028034,True,1.0,20
2,Study 1b (Opus),best_ic,0.076216,0.099334,True,0.1,20
3,Study 1b (Opus),mean_ic,0.029919,0.040215,True,0.1,20


---
##### R6 Effective breadth and R7 capacity
Participation ratio of the 13-group correlation matrices; average daily dollar volume of the proxy ETFs and the AUM at which monthly trading reaches 1% and 5% of it. POST-HOC.

In [51]:
def participation_ratio(frame):
    lam = np.linalg.eigvalsh(frame.dropna().corr().to_numpy())
    return float(lam.sum() ** 2 / (lam ** 2).sum())


s23_r6 = pd.DataFrame([
    ('13 group excess returns', participation_ratio(R['excess'].reindex(full_months))),
    ('13 group relative returns (group minus equal-weight mean)', participation_ratio(R['relative'].reindex(full_months))),
    ('z(W) across groups', participation_ratio(Z['W'].reindex(full_months))),
    ('HQ12 score across groups', participation_ratio(HQ12_SCORE.reindex(full_months)))], columns=['matrix', 'participation_ratio']).set_index('matrix')
G.write_table(s23_r6, 's23_r6_breadth', label=P.POSTHOC, fmt={'participation_ratio': '{:.1f}'})

ADV_FILE = P.table_path('s23_r7_adv')
proxies = pd.read_csv(P.FORWARD / 'etf_proxies.csv')
proxies = proxies[proxies.group.astype(str) != 'hedge'].assign(group=lambda x: x.group.astype(int))
if not ADV_FILE.exists():
    try:
        import yfinance as yf
        tickers = sorted(set(proxies.etf) | {'SPY'})
        px = yf.download(tickers, start='2025-09-25', end='2026-10-01', auto_adjust=False, progress=False)
        dollar = (px['Close'] * px['Volume']).dropna(how='all').tail(252)
        pd.DataFrame({'adv_usd': dollar.mean(), 'days': dollar.notna().sum(), 'last_day': str(dollar.index.max().date())}).rename_axis('etf').to_csv(ADV_FILE)
    except Exception as exc:
        print('R7 not run: no data access', type(exc).__name__)
R7_OK = ADV_FILE.exists()
if R7_OK:
    adv = pd.read_csv(ADV_FILE, index_col=0)
    etf_of = proxies.set_index('group').etf
    rows = []
    for book, weights in [('A3', A3_WEIGHTS), ('HQ12', WEIGHTS['V2d'])]:
        w = weights.reindex(test_m)
        trade = w.diff().abs().mean()                                   # mean monthly one-way trade per unit of capital
        by_etf = trade[list(P.GROUPS)].groupby(etf_of.reindex(list(P.GROUPS)).to_numpy()).sum()
        by_etf['SPY'] = trade['market']
        ratio = (adv.adv_usd.reindex(by_etf.index) / by_etf)            # AUM at which the monthly trade equals one day's volume
        rows.append({'book': book, 'binding_etf': ratio.idxmin(), 'binding_adv_usd_m': float(adv.adv_usd[ratio.idxmin()] / 1e6), 'monthly_trade_share_of_aum': float(by_etf[ratio.idxmin()]),
                     'aum_at_1pct_adv_usd_m': float(0.01 * ratio.min() / 1e6), 'aum_at_5pct_adv_usd_m': float(0.05 * ratio.min() / 1e6),
                     'spy_adv_usd_bn': float(adv.adv_usd['SPY'] / 1e9), 'smallest_etf_adv_usd_m': float(adv.adv_usd.drop('SPY').min() / 1e6), 'smallest_etf': adv.adv_usd.drop('SPY').idxmin()})
    s23_r7 = pd.DataFrame(rows).set_index('book')
    r7_show = pd.DataFrame({'Binding ETF': s23_r7.binding_etf.to_numpy(), 'ADV, USD m': [f'{v:,.1f}' for v in s23_r7.binding_adv_usd_m],
                            'Trade / assets': [f'{v:.1%}' for v in s23_r7.monthly_trade_share_of_aum], 'Assets at 1% ADV': [f'{v:,.1f}' for v in s23_r7.aum_at_1pct_adv_usd_m],
                            'at 5%': [f'{v:,.1f}' for v in s23_r7.aum_at_5pct_adv_usd_m], 'SPY ADV, USD bn': [f'{v:,.0f}' for v in s23_r7.spy_adv_usd_bn]}, index=pd.Index(s23_r7.index, name='Book'))
    G.write_table(s23_r7, 's23_r7_capacity', label=P.POSTHOC, display=r7_show, align='llrrrrr')
s23_r6

,participation_ratio
matrix,
13 group excess returns,1.956523
13 group relative returns (group minus equal-weight mean),9.050167
z(W) across groups,7.360634
HQ12 score across groups,10.106317


---
##### R8 Judge fix and R9 feedback ablation
Model calls are made once by `analysis/llm_runs.py` (original models, headless, every call logged); this section only reads the saved outputs. POST-HOC; R9 uses research data only.

In [52]:
LLM = P.OUT2 / 'llm'
R8_FILE, R9_FILE = P.table_path('s23_r8_judge_pairs'), P.table_path('s23_r9_ablation_proposals')
R8_OK, R9_OK = R8_FILE.exists(), R9_FILE.exists()
if R8_OK:
    r8p = pd.read_csv(R8_FILE)
    # Supplementary reading, outside the spec: accept an answer wrapped in a ```json fence (the spec counts it as undefined).
    log8 = {json.loads(l)['call_id']: json.loads(l) for l in (LLM / 'r8_calls.jsonl').read_text().splitlines()}        # the last record per call wins
    texts = {study: {r['report_id']: r['text'] for r in json.loads((root / 'outputs/agents/migrations.json').read_text())}
             for study, root in [('Study 1 (Sonnet)', P.SONNET), ('Study 1b (Opus)', P.OPUS)]}
    squash = lambda s: ' '.join(str(s).lower().split())
    lenient = []
    for row in r8p.itertuples():
        raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', log8[f'r8_{row.study.split()[1]}_{row.item}']['response'].strip())
        try:
            answer = json.loads(raw)
            ok = isinstance(answer, dict) and isinstance(answer.get('uses_claim'), bool)
            quote = squash(answer.get('quote', '')) if ok else ''
            lenient.append(bool(answer['uses_claim'] and quote not in ('', 'none') and quote in squash(texts[row.study][row.report_id])) if ok else np.nan)
        except ValueError:
            lenient.append(np.nan)
    r8p['machine_after_fence_stripped'] = lenient
    rows = []
    for study, g in r8p.groupby('study', sort=False):
        for name, col in [('original judge', 'machine_before'), ('judge with verbatim quote', 'machine_after'), ('with quote, fenced JSON accepted (supplementary)', 'machine_after_fence_stripped')]:
            d = g[g[col].notna()]
            yes = d[col].astype(str).str.lower().eq('true')
            human = d.human.astype(str).str.lower().eq('true')
            rows.append({'study': study, 'judge': name, 'pairs': len(g), 'defined': len(d), 'agree': int((yes == human).sum()), 'judge_yes': int(yes.sum()),
                         'human_yes': int(g.human.astype(str).str.lower().eq('true').sum()), 'false_yes': int((yes & ~human).sum()), 'missed_yes': int((~yes & human).sum())})
    s23_r8 = pd.DataFrame(rows)
    G.write_table(s23_r8, 's23_r8_judge_summary', index=False, label=P.POSTHOC, align='llrrrrrrr',
                  display=s23_r8.rename(columns={'study': 'Study', 'judge': 'Judge', 'pairs': 'Pairs', 'defined': 'Usable', 'agree': 'Agree with human', 'judge_yes': 'Judge yes', 'human_yes': 'Human yes', 'false_yes': 'False yes', 'missed_yes': 'Missed yes'}))
if R9_OK:
    r9p = pd.read_csv(R9_FILE)
    v9 = r9p[r9p.valid.astype(str).str.lower().eq('true')]
    s23_r9 = pd.DataFrame([{'condition': c, 'seeds': g.seed.nunique(), 'proposals': int((r9p.condition == c).sum()), 'valid': len(g),
                            'regime_proposals': int(g.uses_regime.astype(str).str.lower().eq('true').sum()), 'regime_share': float(g.uses_regime.astype(str).str.lower().eq('true').mean()),
                            'nonmissing_below_half_share': float((g.active_share < 0.5).mean()), 'informative_below_half_share': float((g.informative_share < 0.5).mean()),
                            'best_ic': float(g.mean_ic.max())} for c, g in v9.groupby('condition', sort=False)])
    G.write_table(s23_r9, 's23_r9_ablation_summary', index=False, label=P.POSTHOC, align='lrrrrrrr', display=pd.DataFrame({'Feedback': s23_r9.condition.map({'A_with_tercile_split': 'with tercile split', 'B_without_tercile_split': 'without tercile split'}).to_numpy(),
                  'Seeds': s23_r9.seeds.to_numpy(), 'Proposals': s23_r9.proposals.to_numpy(), 'Valid': s23_r9.valid.to_numpy(), 'With a regime gate': s23_r9.regime_proposals.to_numpy(), 'Share': [f'{v:.0%}' for v in s23_r9.regime_share],
                  'Informative in under half of months': [f'{v:.0%}' for v in s23_r9.informative_below_half_share], 'Best research IC': [f'{v:+.3f}' for v in s23_r9.best_ic]}))
print('R8 saved outputs present:', R8_OK, '| R9 saved outputs present:', R9_OK)
s23_r8 if R8_OK else None

R8 saved outputs present: True | R9 saved outputs present: True


,study,judge,pairs,defined,agree,judge_yes,human_yes,false_yes,missed_yes
0,Study 1 (Sonnet),original judge,8,2,1,1,1,1,0
1,Study 1 (Sonnet),judge with verbatim quote,8,8,7,2,1,1,0
2,Study 1 (Sonnet),"with quote, fenced JSON accepted (supplementary)",8,8,7,2,1,1,0
3,Study 1b (Opus),original judge,17,17,10,10,3,7,0
4,Study 1b (Opus),judge with verbatim quote,17,7,3,5,3,4,0
5,Study 1b (Opus),"with quote, fenced JSON accepted (supplementary)",17,17,9,11,3,8,0


---
##### Report tables: time patterns, turnover and costs
Every strategy over the three windows the brief asks for (full sample, post-2010, last 18 months), and one table of annual turnover, cost drag and break-even cost. Study 1 rows are CONFIRMATORY (saved files); Study 2 rows are POST-HOC.

In [53]:
cw = res['course_windows']
rows = []
for s in ['A0', 'A1', 'A1-T', 'A3', 'A4']:
    rows.append({'strategy': P.STRATEGY_NAMES[s], 'study': 'Study 1', 'label': P.CONFIRMATORY, 'test': var['ASOF'][s]['sharpe'],
                 'full': cw[s]['full']['sharpe'], 'post_2010': cw[s]['post_2010']['sharpe'], 'last_18m': win[s]['last_18']['sharpe']})
for name, old in OLD.items():
    g = fl.xs(old, level='variant')
    rows.append({'strategy': P.STRATEGY_NAMES[name], 'study': 'Study 2', 'label': P.POSTHOC, 'test': float(g.loc['test', 'sharpe']),
                 'full': float(g.loc['full', 'sharpe']), 'post_2010': float(g.loc['post-2010', 'sharpe']), 'last_18m': float(g.loc['last 18m', 'sharpe'])})
s23_time = pd.DataFrame(rows)
time_show = pd.DataFrame({'Strategy': s23_time.strategy.to_numpy(), 'Study': s23_time.study.to_numpy(), 'Test': [f'{v:+.2f}' for v in s23_time.test],
                          'Full sample': [f'{v:+.2f}' for v in s23_time.full], 'Post-2010': [f'{v:+.2f}' for v in s23_time.post_2010], 'Last 18 months': [f'{v:+.2f}' for v in s23_time.last_18m]})
G.write_table(s23_time, 's23_time_patterns', index=False, label=f'{P.CONFIRMATORY} (Study 1) / {P.POSTHOC} (Study 2)', display=time_show, align='llrrrr')

rows = [{'book': P.STRATEGY_NAMES['A3'], 'annual_turnover': 12 * var['ASOF']['A3']['turnover'], 'cost_drag_pa': 12 * var['ASOF']['A3']['cost'],
         'gross_pa': var['ASOF']['A3']['mean_gross_annual'], 'net_pa': var['ASOF']['A3']['mean_net_annual'], 'breakeven_bp': res['cost_breakeven_bps'], 'breakeven_method': 'Study 1 results.json'}]
for name, old in OLD.items():
    l10, l0 = LEDGERS[(old, 10)].reindex(test_m), LEDGERS[(old, 0)].reindex(test_m)
    rows.append({'book': P.STRATEGY_NAMES[name], 'annual_turnover': 12 * l10.turnover.mean(), 'cost_drag_pa': 12 * l10.cost.mean(), 'gross_pa': 12 * l10.gross.mean(),
                 'net_pa': 12 * l10.net.mean(), 'breakeven_bp': 1e4 * l0.net.sum() / l0.turnover.sum(), 'breakeven_method': 'zero-cost run'})
for book in ['A3', 'HQ12']:
    l = ENGINE[(book, 2.0, 'rank')][0].reindex(test_m)
    rows.append({'book': f'{book} rank-weighted', 'annual_turnover': 12 * l.turnover.mean(), 'cost_drag_pa': 12 * l.cost.mean(), 'gross_pa': 12 * l.gross.mean(), 'net_pa': 12 * l.net.mean(),
                 'breakeven_bp': 1e4 * (l.net.sum() + l.turnover.sum() * 10 / 1e4) / l.turnover.sum(), 'breakeven_method': 'approximated from the 10 bp run'})
s23_costs = pd.DataFrame(rows)
cost_show = pd.DataFrame({'Book (test window)': s23_costs.book.to_numpy(), 'Turnover p.a.': [f'{v:.1f}x' for v in s23_costs.annual_turnover], 'Cost drag p.a.': [f'{v:.2%}' for v in s23_costs.cost_drag_pa],
                          'Gross p.a.': [f'{v:+.1%}' for v in s23_costs.gross_pa], 'Net p.a.': [f'{v:+.1%}' for v in s23_costs.net_pa], 'Break-even (bp)': [f'{v:+.0f}' for v in s23_costs.breakeven_bp]})
G.write_table(s23_costs, 's23_costs_turnover', index=False, label=f'{P.CONFIRMATORY} (A3) / {P.POSTHOC}', display=cost_show, align='lrrrrr')
cost_show

,Book (test window),Turnover p.a.,Cost drag p.a.,Gross p.a.,Net p.a.,Break-even (bp)
0,"A3 agent features, independent (primary)",11.0x,1.11%,-3.5%,-4.6%,-32
1,W wage growth (declared headline),4.7x,0.47%,+2.1%,+1.7%,+46
2,SC signed composite,8.7x,0.88%,-0.5%,-1.3%,-5
3,W+MOM wage growth + momentum,5.8x,0.59%,+3.6%,+3.1%,+62
4,"HQ12 hires + quits reversal, 12-month hold",4.4x,0.44%,+3.0%,+2.6%,+68
5,A3 rank-weighted,10.3x,1.05%,-2.5%,-3.6%,-24
6,HQ12 rank-weighted,4.4x,0.45%,+3.0%,+2.6%,+69


---
##### Study 2.3 report and Gate 4
Write `study1/robustness.md` and `study2/robustness_2_3.md`, record the first-run hashes of the new tables, and check that the spec predates them and Study 1's folder is unchanged.

In [54]:
S23_FILES = ['d11_study1_robustness', 'd11_a3_horizon_profile', 's23_r1_validation', 's23_r1_variants', 's23_r1_overlap', 's23_r1_sc_signs', 's23_r1_hq12_battery', 's23_r1_hq12_dropone',
             's23_r2_risk_overlay', 's23_r2_distribution', 's23_r3_rank_weighted', 's23_r4_agents_by_seed', 's23_r4_permutation', 's23_r5_tokens', 's23_r5_tokens_ic', 's23_r6_breadth', 's23_time_patterns', 's23_costs_turnover']
s23_log = P.OUT2 / 'study2_3_run_log.md'
S23_NOW = utc_now()
if not s23_log.exists():
    s23_log.write_text('# Study 2.3 run log\n\nThe items of `analysis/specs/study2_3_robustness.md` that need no model call or download ran once, at **' + S23_NOW + '** (spec frozen ' + S23_UTC + '). '
                       'Later executions must reproduce these tables byte for byte.\n\nFIRST_RUN_UTC ' + S23_NOW + '\n\n| table | sha256 of first run |\n|---|---|\n')
logged = dict(re.findall(r'\| `([\w]+)` \| `([0-9a-f]{64})` \|', s23_log.read_text()))
for name, digest in logged.items():
    if sha256(P.table_path(name)) != digest:
        stop(f'Study 2.3 output {name} differs from the first run recorded in study2_3_run_log.md')
with open(s23_log, 'a') as handle:
    for name in S23_FILES:
        if name not in logged:
            handle.write(f'| `{name}` | `{sha256(P.table_path(name))}` |\n')
S23_FIRST_UTC = re.search(r'FIRST_RUN_UTC (\S+)', s23_log.read_text()).group(1)

md = lambda frame, **kw: G.markdown(frame, **kw)
(P.OUT1 / 'robustness.md').write_text('\n'.join([
    '# Study 1 robustness: A3 (agent features, independent; primary) and A0 (fixed rule)\n',
    f'Regenerated {S23_NOW} by analysis/run.ipynb. Rows labelled CONFIRMATORY are read from Study 1\'s saved `results.json` and `research_summary.json` and are unchanged; rows labelled POST-HOC are '
    'diagnostics added in Study 2.3 (spec `analysis/specs/study2_3_robustness.md`, section 1) and cannot change the verdict: **Do not implement**.\n',
    '## Consolidated table\n', 'Source: `analysis/output/study1/tables/d11_study1_robustness.csv`.\n', md(d11[['item', 'A3', 'A0', 'label', 'source']], index=False), '',
    '## Horizon profile of the A3 score (POST-HOC)\n', 'Source: `d11_a3_horizon_profile.csv`; figure `report/figures/fig_s1_horizon.pdf`.\n', md(d11_h.round(4), index=False), '',
    '## Freeze record (CONFIRMATORY)\n', 'Source: `d11_freeze_check.csv`, computed from `FREEZE.json`, `evaluation_started.json` and the files in `study1_preregistered/`.\n', md(freeze, index=False), '']))

parts = ['# Study 2.3: additional robustness (POST-HOC)\n',
         f'First run **{S23_FIRST_UTC}** (regenerated {S23_NOW}); spec `analysis/specs/study2_3_robustness.md` frozen **{S23_UTC}** (sha256 `{S23_HASH}`). Deflated Sharpe ratios here use N = {N_TRIALS_23} looks. '
         'Names: W, SC, W+MOM, HQ12 are the variants stored as V2a, V2b, V2c, V2d in the Study 2.1 and 2.2 result files. Nothing here changes the confirmatory verdict (A3, Do not implement).\n',
         '## R1 As-known rerun\n', f'Validation: {"passed" if R1_OK else "FAILED"} (`s23_r1_validation.csv`). Reading fixed in the spec: **{R1_READING}**.\n', md(ak_check, index=False), '',
         'Test window, fixed lag (Study 2.1) against as known (`s23_r1_variants.csv`, `s23_r1_overlap.csv`):\n', md(r1_show), '',
         'Sharpe by window, as known:\n', md(s23_r1.pivot(index='variant', columns='window', values='sharpe_asknown')[WINDOW_NAMES].reindex(list(V23)).round(3)), '',
         'HQ12 battery (`s23_r1_hq12_battery.csv`):\n', md(s23_bat.round(3)), '',
         '## R2 Risk overlay\n', 'Source: `s23_r2_risk_overlay.csv`, `s23_r2_distribution.csv`; figure `fig_s23_gross_needed.pdf`.\n', md(r2o_show, index=False), '', md(s23_r2d.round(3)), '',
         '## R3 Rank-weighted books\n', 'Source: `s23_r3_rank_weighted.csv`.\n', md(r3o_show, index=False), '',
         '## R4 Agents: arm gap against seed spread\n', 'Source: `s23_r4_agents_by_seed.csv`, `s23_r4_permutation.csv`; figure `fig_s23_agents_seeds.pdf`.\n', md(r4_show4, index=False), '', md(s23_r4p.round(3), index=False), '',
         '## R5 Tokens\n', ('Source: `s23_r5_tokens.csv`, `s23_r5_tokens_ic.csv`.\n' if R5_OK else 'Not run: no usage in the log.\n'), md(s23_r5.round(0), index=False), '', md(s23_r5r.round(2), index=False), '',
         '## R6 Effective breadth\n', 'Source: `s23_r6_breadth.csv`.\n', md(s23_r6.round(2)), '',
         '## Time patterns and costs (report tables)\n', md(time_show, index=False), '', md(cost_show, index=False), '',
         '## R7 Capacity\n', (md(s23_r7.round(3)) if R7_OK else 'Not run: no data access.'), '',
         '## R8 Judge fix\n', (md(s23_r8, index=False) if R8_OK else 'Not run (no saved output from analysis/llm_runs.py).'), '',
         '## R9 Feedback ablation\n', (md(s23_r9.round(3), index=False) if R9_OK else 'Not run (no saved output from analysis/llm_runs.py).'), '']
(P.OUT2 / 'robustness_2_3.md').write_text('\n'.join(parts))

fingerprint_end, n_end = frozen_fingerprint()
checks4 = [('Study 2.3 spec frozen before the first Study 2.3 output', f'{S23_UTC} < {S23_FIRST_UTC}', datetime.fromisoformat(S23_UTC) < datetime.fromisoformat(S23_FIRST_UTC)),
           ('Study 2.3 spec unchanged since freezing', S23_HASH[:16], sha256(P.SPECS / 'study2_3_robustness.md') == S23_HASH),
           ('Variant engine reproduces the Study 1 backtest', f'max difference {engine_error:.1e}', ENGINE_OK),
           ('As-known inputs validated against Study 1 saved files', 's23_r1_validation.csv', R1_OK),
           ('Study 1 folder unchanged', f'{n_end} files, {fingerprint_end[:12]}...', (fingerprint_end, n_end) == P.FROZEN_FINGERPRINT)]
gate4 = pd.DataFrame(checks4, columns=['check', 'value', 'pass'])
passed4 = bool(gate4['pass'].all())
(P.OUT2 / 'gate4.md').write_text(f'# Gate 4\n\nRun {utc_now()} by analysis/run.ipynb. **Result: {"PASS" if passed4 else "FAIL"}**\n\n' + G.markdown(gate4, index=False) + '\n')
if not passed4:
    stop('Gate 4 failed. See analysis/output/study2/gate4.md:\n\n' + G.markdown(gate4, index=False))
print('Gate 4:', 'PASS' if passed4 else 'FAIL', '| R1 reading:', R1_READING)
gate4

Gate 4: PASS | R1 reading: as-known confirms the fixed-lag reading


,check,value,pass
0,Study 2.3 spec frozen before the first Study 2...,2026-10-04T09:18:39+00:00 < 2026-10-04T09:26:2...,True
1,Study 2.3 spec unchanged since freezing,068a3e0396332514,True
2,Variant engine reproduces the Study 1 backtest,max difference 9.9e-17,True
3,As-known inputs validated against Study 1 save...,s23_r1_validation.csv,True
4,Study 1 folder unchanged,"3067 files, eb109be96917...",True
